# OpenBind Blind Challenge 1, Zika NS2B-NS3 protease pose prediction

This notebook holds the whole pipeline in one place. Each section starts with a short note on what the next cell does.

**Goal.** Predict one protein-ligand complex for each of 400 compounds and submit 400 PDB files in one ZIP.
A pose succeeds only if BiSyRMSD is below 1.5 A, lDDT-PLI is above 0.8, and all PoseBusters checks pass.

**Deadlines.** Interim leaderboard 11 Nov 2026, final 16 Dec 2026 (both 23:59 UTC).

**Environments.** This notebook runs in `zika`. Tools that need a different stack are called through `run_in_env`.

| Env | Holds | Used for |
|---|---|---|
| `zika` | Boltz, PoseBusters, GNINA, Vina, RDKit, Meeko, torch | generating and checking poses (this kernel) |
| `zika-eval` | OpenStructure | the official scorer |
| `zika-md` | OpenMM, PDBFixer | pocket relaxation (CPU only) |

**Only one generated file.** The long Boltz batch must outlive this kernel, so section 6 writes a small bash script (`run_batch.sh`) and starts it in the background. Everything else is visible here.

## 1. Parameters
Set the values in the panel below, then click **Apply** before running the next cells. If you do not click Apply, the defaults in the cell are used. The cell is tagged `parameters` so papermill can override the values when the notebook runs headless.

| Parameter | What it does |
|---|---|
| `RUN` | Name of the run. Everything is saved under `runs/<RUN>/`. Use a new name for each experiment so results are not overwritten. |
| `LIMIT` | How many compounds to build inputs for. "All 400" is the full set. A smaller number gives a quick test. |
| `SAMPLES` | How many structures Boltz generates per compound. More samples give more chances that one passes the PoseBusters checks, but the run takes proportionally longer. |
| `CHUNK` | How many compounds go into each Boltz call. 400 compounds at 50 per chunk makes 8 chunks. Finished chunks are skipped on a rerun, so an interruption loses at most one chunk. |
| `SEED` | Random seed. The same seed reproduces the same structures. A different seed gives different samples. |
| `MIN_WSL_GB` | Memory check. The batch refuses to start if WSL has less memory than this, because the pilot run was killed at the 6 GB cap. |
| `ALLOW_LOW_MEMORY` | Switches off the memory check when ticked. Leave it unticked unless you are testing on purpose. |

If you change a value after running the setup cell, click Apply again and rerun the setup cell.

In [ ]:
# Defaults. Papermill can override these when run headless.
RUN = "baseline"
LIMIT = None
SAMPLES = 5
CHUNK = 50
SEED = 42
MIN_WSL_GB = 9
ALLOW_LOW_MEMORY = False

# Interactive panel. Set values, then click Apply.
import ipywidgets as w
from IPython.display import display

_style = {"description_width": "150px"}
_OTHER = "__other__"


def _lay():
    return w.Layout(width="360px")


def _choice(label, presets, current, none_label=None, max_value=10**6):
    """Dropdown with presets, the current default, and an Other option that reveals a number box."""
    presets = sorted(set(presets + ([current] if isinstance(current, int) else [])))
    options = []
    if none_label:
        options.append((none_label, None))
    options += [(str(p), p) for p in presets]
    options.append(("Other", _OTHER))
    dd = w.Dropdown(options=options, value=current, description=label, style=_style, layout=_lay())
    box = w.BoundedIntText(value=max(1, current) if isinstance(current, int) else 1,
                           min=1, max=max_value, description="enter value", style=_style, layout=_lay())
    box.layout.display = "none"

    def _toggle(change):
        if change["new"] == _OTHER and isinstance(change["old"], int):
            box.value = max(1, change["old"])
        box.layout.display = "" if change["new"] == _OTHER else "none"

    dd.observe(_toggle, names="value")

    def get():
        return box.value if dd.value == _OTHER else dd.value

    return w.VBox([dd, box]), get


_run = w.Text(value=RUN, description="RUN", style=_style, layout=_lay())
_limit_ui, _limit = _choice("LIMIT", [5, 10, 25, 50, 100, 200], LIMIT, none_label="All 400", max_value=400)
_samples_ui, _samples = _choice("SAMPLES", [1, 2, 3, 5, 10, 20, 50], SAMPLES)
_chunk_ui, _chunk = _choice("CHUNK", [10, 25, 50, 100], CHUNK)
_seed = w.IntText(value=SEED, description="SEED", style=_style, layout=_lay())
_mem = w.IntSlider(value=MIN_WSL_GB, min=4, max=30, description="MIN_WSL_GB", style=_style, layout=_lay())
_low = w.Checkbox(value=ALLOW_LOW_MEMORY, description="ALLOW_LOW_MEMORY", style=_style)
_btn = w.Button(description="Apply", button_style="primary")
_out = w.Output()


def _apply(_):
    global RUN, LIMIT, SAMPLES, CHUNK, SEED, MIN_WSL_GB, ALLOW_LOW_MEMORY
    RUN = _run.value.strip().replace(" ", "_") or "baseline"
    LIMIT, SAMPLES, CHUNK = _limit(), _samples(), _chunk()
    SEED, MIN_WSL_GB, ALLOW_LOW_MEMORY = _seed.value, _mem.value, _low.value
    with _out:
        _out.clear_output()
        print(f"RUN={RUN} LIMIT={LIMIT} SAMPLES={SAMPLES} CHUNK={CHUNK} SEED={SEED} MIN_WSL_GB={MIN_WSL_GB} ALLOW_LOW_MEMORY={ALLOW_LOW_MEMORY}")


_btn.on_click(_apply)
display(w.VBox([_run, _limit_ui, _samples_ui, _chunk_ui, _seed, _mem, _low, _btn, _out]))

## 2. Setup and paths
Imports, folder locations, and small helpers used everywhere.
`run_in_env` runs a command inside another conda env and returns the result.
`wsl_memory_gb` reads how much memory WSL actually has, which matters because a 6 GB cap killed the pilot run.

In [ ]:
import os, sys, re, json, shutil, subprocess, zipfile, datetime, inspect, fnmatch
from pathlib import Path
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
import ipywidgets as w
from IPython.display import display
try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = lambda x, **k: x


# ---------- small helpers ----------
def fix_wsl_path(p):
    """Convert //wsl.localhost/Ubuntu/... paths from Windows dialogs to Linux paths."""
    p = str(p).replace("\\", "/")
    m = re.match(r"^//wsl(?:\.localhost|\$)/[^/]+(/.*)$", p)
    return Path(m.group(1)) if m else Path(p)


def wsl_memory_gb():
    """Total memory visible to WSL, in GB."""
    with open("/proc/meminfo") as f:
        kb = int(next(line for line in f if line.startswith("MemTotal")).split()[1])
    return kb / 1024 / 1024


def run_in_env(env, args, **kw):
    """Run a command inside another conda env, for example run_in_env('zika-eval', ['python', 'x.py'])."""
    return subprocess.run(["conda", "run", "-n", env, *args], capture_output=True, text=True, **kw)


# ---------- Windows popups (PowerShell) ----------
def to_windows(p):
    """Linux path to Windows path, so a popup can open there."""
    try:
        return subprocess.run(["wslpath", "-w", str(p)], capture_output=True, text=True).stdout.strip()
    except FileNotFoundError:
        return ""


def to_linux(text):
    """Windows or Linux path text to a Linux Path."""
    text = str(text).strip()
    if re.match(r"^[A-Za-z]:\\|^\\\\", text):
        return Path(subprocess.run(["wslpath", "-u", text], capture_output=True, text=True).stdout.strip())
    return Path(text).expanduser()


def _ps_quote(s):
    return str(s).replace("'", "''")


def pick_path(kind, title, start, filt="All files|*.*"):
    """Open a Windows dialog. kind is 'file' or 'dir'. Returns a Linux Path, or None if cancelled."""
    start_w = to_windows(start) if start else ""
    if kind == "file":
        extra = f"$d.Filter = '{_ps_quote(filt)}'; "
    else:
        extra = ("$d.ValidateNames = $false; $d.CheckFileExists = $false; $d.CheckPathExists = $true; "
                 "$d.FileName = 'Select this folder'; ")
    script = ("Add-Type -AssemblyName System.Windows.Forms; "
              "$f = New-Object System.Windows.Forms.Form; $f.TopMost = $true; "
              "$d = New-Object System.Windows.Forms.OpenFileDialog; "
              f"$d.Title = '{_ps_quote(title)}'; $d.InitialDirectory = '{_ps_quote(start_w)}'; " + extra +
              "if ($d.ShowDialog($f) -eq 'OK') { Write-Output $d.FileName }")
    try:
        out = subprocess.run(["powershell.exe", "-NoProfile", "-STA", "-Command", script],
                             capture_output=True, text=True).stdout.strip()
    except FileNotFoundError:
        return None
    if not out:
        return None
    p = to_linux(out)
    if kind == "dir":
        return p if p.is_dir() else p.parent
    return p


# ---------- search ----------
_SKIP = {".git", "__pycache__", ".ipynb_checkpoints", "out", "chunks", "node_modules"}


def search_paths(term, root, kind="file", limit=200):
    """Find files (or folders) under root whose name contains term, or matches a *pattern*."""
    term = term.strip().lower()
    if not term or root is None or not Path(root).is_dir():
        return []
    use_glob = any(c in term for c in "*?")
    hits = []
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [d for d in dirnames if d not in _SKIP]
        for name in (filenames if kind == "file" else dirnames):
            low = name.lower()
            if fnmatch.fnmatch(low, term) if use_glob else term in low:
                hits.append(Path(dirpath) / name)
                if len(hits) >= limit:
                    return hits
    return hits


# ---------- one path = text box + Browse popup + search ----------
_pstyle = {"description_width": "170px"}


class PathPicker:
    """A path field with a Windows Browse popup and a name search that fills a dropdown of matches."""

    def __init__(self, label, kind="file", filt="All files|*.*", root_fn=None, default=None):
        self.kind, self.filt, self.label = kind, filt, label
        self.root_fn = root_fn or (lambda: Path.home())
        self.text = w.Text(value=str(default) if default else "", description=label, placeholder="browse, search, or type a path",
                           style=_pstyle, layout=w.Layout(width="640px"))
        self.browse = w.Button(description="Browse...", layout=w.Layout(width="95px"))
        self.term = w.Text(placeholder="search name or *pattern*", layout=w.Layout(width="270px"))
        self.find = w.Button(description="Search", layout=w.Layout(width="85px"))
        self.msg = w.HTML()
        self.hits = w.Dropdown(options=[], layout=w.Layout(width="640px", display="none"))
        self.browse.on_click(self._on_browse)
        self.find.on_click(self._on_find)
        self.hits.observe(self._on_hit, names="value")

    @property
    def path(self):
        t = self.text.value.strip()
        return to_linux(t) if t else None

    def _on_browse(self, _):
        cur = self.path
        start = cur if cur is not None and cur.exists() else self.root_fn()
        start = start if Path(start).is_dir() else Path(start).parent
        got = pick_path(self.kind, f"Select: {self.label}", start, self.filt)
        if got is None:
            self.msg.value = "no selection (or powershell.exe not available, use Search or type the path)"
        else:
            self.msg.value = ""
            self.text.value = str(got)

    def _on_find(self, _):
        root = Path(self.root_fn())
        found = search_paths(self.term.value, root, self.kind)
        self.hits.options = [(f"choose a match ({len(found)} found)", "")] + [(str(p.relative_to(root)), str(p)) for p in found]
        self.hits.value = ""
        self.hits.layout.display = "" if found else "none"
        self.msg.value = "" if found else "no match"

    def _on_hit(self, change):
        if change["new"]:
            self.text.value = change["new"]

    @property
    def widget(self):
        return w.VBox([w.HBox([self.text, self.browse]), w.HBox([self.term, self.find, self.msg]), self.hits],
                      layout=w.Layout(margin="0 0 12px 0"))


# ---------- the pickers ----------
def _default_root():
    """The folder above the notebooks folder, or the current folder."""
    cwd = Path.cwd()
    return cwd.parent if cwd.name == "notebooks" else cwd


_root_p = PathPicker("Project folder", kind="dir", root_fn=lambda: Path.home(), default=_default_root())


def _project():
    return _root_p.path or _default_root()


def _auto(pattern):
    hits = search_paths(pattern, _project(), "file", limit=1)
    return hits[0] if hits else None


def _auto_tutorial():
    hits = search_paths("evaluate_predictions.py", _project(), "file", limit=1)
    return hits[0].parent.parent if hits else None


_csv_p = PathPicker("Challenge CSV", filt="CSV files|*.csv|All files|*.*", root_fn=_project, default=_auto("*TEST-BLINDED.csv"))
_fasta_p = PathPicker("NS2B-NS3 FASTA", filt="FASTA files|*.fasta;*.fa|All files|*.*", root_fn=_project, default=_auto("*NS2B-NS3*.fasta"))
_ns2b_p = PathPicker("MSA, NS2B (chain A)", filt="CSV files|*.csv|All files|*.*", root_fn=_project, default=_auto("zika_ns2b.csv"))
_ns3_p = PathPicker("MSA, NS3 (chain B)", filt="CSV files|*.csv|All files|*.*", root_fn=_project, default=_auto("zika_ns3.csv"))
_tut_p = PathPicker("Tutorial repo folder", kind="dir", root_fn=_project, default=_auto_tutorial())

ENV_BIN = Path(sys.executable).parent
_status = w.Output()


def set_paths(_=None):
    """Read the pickers, check the paths exist, and set the globals the other cells use."""
    global ROOT, RUNDIR, SCRATCH, TUTORIAL, CSV, FASTA, MSA_NS2B, MSA_NS3
    ROOT, TUTORIAL, CSV, FASTA = _root_p.path, _tut_p.path, _csv_p.path, _fasta_p.path
    MSA_NS2B, MSA_NS3 = _ns2b_p.path, _ns3_p.path
    items = [("Project folder", ROOT), ("Challenge CSV", CSV), ("NS2B-NS3 FASTA", FASTA),
             ("MSA, NS2B", MSA_NS2B), ("MSA, NS3", MSA_NS3), ("Tutorial repo folder", TUTORIAL)]
    with _status:
        _status.clear_output()
        for name, p in items:
            print(f"{'OK     ' if p is not None and p.exists() else 'MISSING'} {name}: {p}")
        if ROOT is not None and ROOT.is_dir():
            RUNDIR = ROOT / "runs" / RUN
            SCRATCH = ROOT / "runs" / "_scratch"
            SCRATCH.mkdir(parents=True, exist_ok=True)
            print("run folder:", RUNDIR)
        print("python:", sys.executable)


_confirm = w.Button(description="Confirm paths", button_style="primary")
_confirm.on_click(set_paths)
display(w.VBox([_root_p.widget, _csv_p.widget, _fasta_p.widget, _ns2b_p.widget, _ns3_p.widget, _tut_p.widget, _confirm, _status]))
set_paths()

## 3. Environment health check
Confirms this kernel is the right env, that the GPU is visible, and how much memory WSL has.
Then `health_check` tests every tool in all three envs and prints a PASS or FAIL table.
This replaces `setup_check.py`. Run it after any install.

In [ ]:
import torch

print("env:", ENV_BIN.parent.name)
print("cuda available:", torch.cuda.is_available())
print(f"WSL memory: {wsl_memory_gb():.1f} GB (batch needs {MIN_WSL_GB}+)")
print("boltz:", shutil.which("boltz"), "| bust:", shutil.which("bust"), "| vina:", shutil.which("vina"))


def health_check():
    checks = [
        ("zika", "chem stack", 'python -c "import rdkit, meeko, Bio, pandas, sklearn"'),
        ("zika", "numpy 1.26", "python -c \"import numpy; assert numpy.__version__.startswith('1.26')\""),
        ("zika", "torch cuda", 'python -c "import torch; assert torch.cuda.is_available()"'),
        ("zika", "boltz cli", "boltz --help"),
        ("zika", "posebusters", "bust --help"),
        ("zika", "vina", "vina --version"),
        ("zika", "fpocket", "which fpocket"),
        ("zika", "gnina", "$HOME/gnina --version"),
        ("zika-eval", "openstructure", 'python -c "from ost.mol.alg.ligand_scoring_scrmsd import SCRMSDScorer; from ost.mol.alg.ligand_scoring_lddtpli import LDDTPLIScorer"'),
        ("zika-md", "openmm cpu", "python -c \"import openmm, pdbfixer; from openmm import Platform; Platform.getPlatformByName('CPU')\""),
    ]
    rows = []
    for env, name, cmd in checks:
        r = run_in_env(env, ["bash", "-c", cmd])
        rows.append({"env": env, "check": name, "result": "PASS" if r.returncode == 0 else "FAIL"})
    table = pd.DataFrame(rows)
    print(f"{(table.result == 'PASS').sum()}/{len(table)} passed")
    return table


# Uncomment to run (takes about a minute).
# health_check()

## 4. Challenge data
Loads the 400 compound SMILES and the NS2B and NS3 sequences from the tutorial FASTA.
The asserts catch mistakes early. NS2B must be 46 residues (chain A) and NS3 168 (chain B), matching the organizers' report.
The tutorial's own `blind_challenge_001.yaml` merged the two chains by mistake, so we always read the FASTA directly.

In [ ]:
def read_fasta(path):
    seqs = {}
    for rec in Path(path).read_text().split(">")[1:]:
        lines = rec.strip().split("\n")
        seqs[lines[0].split("|")[1]] = "".join(lines[1:])
    return seqs


seqs = read_fasta(FASTA)
NS2B, NS3 = seqs["Chain A"], seqs["Chain B"]
assert (len(NS2B), len(NS3)) == (46, 168), (len(NS2B), len(NS3))

df = pd.read_csv(CSV)
assert len(df) == 400 and df["Molecule_Name"].is_unique
mols = [Chem.MolFromSmiles(s) for s in df["SMILES"]]
bad = [n for n, m in zip(df["Molecule_Name"], mols) if m is None]
assert not bad, f"unparseable SMILES: {bad}"
df["heavy_atoms"] = [m.GetNumHeavyAtoms() for m in mols]

# The saved MSAs must start with the query sequence.
for msa, seq in [(MSA_NS2B, NS2B), (MSA_NS3, NS3)]:
    assert pd.read_csv(msa).loc[0, "sequence"] == seq, f"MSA does not match its sequence: {msa}"

print("NS2B", len(NS2B), "| NS3", len(NS3), "| compounds", len(df))
print(df["heavy_atoms"].describe().round(1).to_string())
df.head()

## 5. Boltz inputs
Builds one Boltz YAML per compound. Each has the two protein chains (A is NS2B, B is NS3) with the saved MSA files, plus the ligand as chain C.
Using saved MSAs means the ColabFold server is not called again.

In [ ]:
def boltz_yaml(smiles):
    return "\n".join([
        "version: 1",
        "sequences:",
        "  - protein:",
        "      id: A",
        f"      sequence: {NS2B}",
        f"      msa: {MSA_NS2B}",
        "  - protein:",
        "      id: B",
        f"      sequence: {NS3}",
        f"      msa: {MSA_NS3}",
        "  - ligand:",
        "      id: C",
        f"      smiles: '{smiles}'",
        "",
    ])


def build_inputs(frame, run, limit=None):
    out = ROOT / "runs" / run / "inputs"
    out.mkdir(parents=True, exist_ok=True)
    sub = frame.head(limit) if limit else frame
    for name, smi in zip(sub["Molecule_Name"], sub["SMILES"]):
        (out / f"{name}.yaml").write_text(boltz_yaml(smi))
    return len(sub)


n_inputs = build_inputs(df, RUN, LIMIT)
print(n_inputs, "inputs written to", RUNDIR / "inputs")

## 6. Boltz batch in chunks
Splits the inputs into chunks (folders of symlinks) and starts a background bash loop that runs `boltz predict` on one chunk at a time.
A chunk counts as finished when all its PDB files exist, and it then gets a `DONE` marker. A rerun skips finished chunks, so an interruption costs at most one chunk.
The loop judges success by counting files, not by exit code, because the pilot printed `Killed` after the results were already saved.
`launch_batch` refuses to start if WSL has too little memory or another batch is already running.

In [ ]:
def prepare_chunks(run, size):
    inputs = sorted((ROOT / "runs" / run / "inputs").glob("*.yaml"))
    for k in range(0, len(inputs), size):
        cdir = ROOT / "runs" / run / "chunks" / f"chunk_{k // size + 1:02d}"
        cdir.mkdir(parents=True, exist_ok=True)
        for f in inputs[k:k + size]:
            link = cdir / f.name
            if not link.exists():
                link.symlink_to(f)
    return len(list((ROOT / "runs" / run / "chunks").glob("chunk_*")))


def write_batch_script(run, samples, seed):
    rundir = ROOT / "runs" / run
    script = rundir / "run_batch.sh"
    script.write_text(f"""#!/usr/bin/env bash
cd {rundir}
for c in chunks/chunk_*; do
  n=$(basename "$c")
  out="out/$n"
  if [ -f "$out/DONE" ]; then echo "skip $n"; continue; fi
  boltz predict "$c" --out_dir "$out" --diffusion_samples {samples} --output_format pdb --seed {seed} --num_workers 1
  got=$(find "$out" -name '*_model_*.pdb' | wc -l)
  want=$(( $(ls "$c"/*.yaml | wc -l) * {samples} ))
  echo "$n pdbs=$got/$want"
  if [ "$got" -ge "$want" ]; then touch "$out/DONE"; else echo "incomplete, rerun to resume"; exit 1; fi
done
echo ALL CHUNKS DONE
""")
    return script


def launch_batch(run, samples, seed):
    if wsl_memory_gb() < MIN_WSL_GB and not ALLOW_LOW_MEMORY:
        raise RuntimeError(f"WSL has {wsl_memory_gb():.1f} GB. Restart WSL after setting memory=10GB in .wslconfig.")
    running = subprocess.run(["pgrep", "-f", "run_batch.sh"], capture_output=True, text=True).stdout.strip()
    if running:
        raise RuntimeError(f"A batch is already running (pid {running}).")
    script = write_batch_script(run, samples, seed)
    env = os.environ.copy()
    env["PATH"] = f"{ENV_BIN}:{env['PATH']}"
    log = open(ROOT / "runs" / run / "run.log", "a")
    p = subprocess.Popen(["bash", str(script)], stdout=log, stderr=subprocess.STDOUT, start_new_session=True, env=env)
    return p.pid


n_chunks = prepare_chunks(RUN, CHUNK)
print(n_chunks, "chunks prepared")

### Start the batch
This is the only cell that starts a long GPU job. Set Windows sleep to "never" first.
It runs in the background, so closing the notebook does not stop it. To resume after an interruption, run this cell again.

In [ ]:
# Uncomment to start.
# pid = launch_batch(RUN, SAMPLES, SEED)
# print("started, pid", pid)

### Progress
Shows how many chunks are finished, how many PDB files exist, and the last lines of the log. Run it whenever you want a status.

In [ ]:
def progress(run, samples=SAMPLES):
    base = ROOT / "runs" / run
    out = base / "out"
    n_chunks = len(list((base / "chunks").glob("chunk_*")))
    done = len(list(out.glob("chunk_*/DONE")))
    pdbs = len(list(out.rglob("*_model_*.pdb")))
    n_inputs = len(list((base / "inputs").glob("*.yaml")))
    print(f"chunks done {done}/{n_chunks} | PDB files {pdbs}/{n_inputs * samples}")
    log = base / "run.log"
    if log.exists():
        print("\n".join(log.read_text().splitlines()[-3:]))


progress(RUN)

## 7. PoseBusters check for every sample
For each Boltz sample this cell splits the complex into protein and ligand, rebuilds the ligand bond orders from the SMILES (a PDB file has none), and runs PoseBusters.
The result per sample is how many of the 22 checks passed, which ones failed, and Boltz's own confidence score.
`score_run` saves results to a CSV after every compound, so it resumes if interrupted.

In [ ]:
def find_pred_dir(run, compound):
    hits = list((ROOT / "runs" / run / "out").glob(f"*/boltz_results_*/predictions/{compound}"))
    return hits[0] if hits else None


def split_complex(pdb_path, tag):
    recs = [l for l in open(pdb_path) if l.startswith(("ATOM", "HETATM"))]
    lig = SCRATCH / f"{tag}_lig.pdb"
    prot = SCRATCH / f"{tag}_prot.pdb"
    lig.write_text("".join(l for l in recs if l[17:20].strip() == "LIG"))
    prot.write_text("".join(l for l in recs if l[17:20].strip() != "LIG"))
    return lig, prot


def ligand_sdf(smiles, lig_pdb, sdf_path):
    mol = AllChem.AssignBondOrdersFromTemplate(Chem.MolFromSmiles(smiles), Chem.MolFromPDBFile(str(lig_pdb)))
    w = Chem.SDWriter(str(sdf_path))
    w.write(mol)
    w.close()


def posebusters(pdb_path, smiles, tag="pb"):
    """Return how many PoseBusters checks pass for one complex."""
    lig, prot = split_complex(pdb_path, tag)
    sdf = SCRATCH / f"{tag}_lig.sdf"
    ligand_sdf(smiles, lig, sdf)
    out_csv = SCRATCH / f"{tag}.csv"
    subprocess.run(["bust", str(sdf), "-p", str(prot), "--outfmt", "csv", "--output", str(out_csv)], capture_output=True, text=True)
    d = pd.read_csv(out_csv)
    checks = d.select_dtypes(include="bool")
    failed = [c for c in checks.columns if not checks[c].all()]
    return {"n_pass": int(checks.iloc[0].sum()), "n_total": int(checks.shape[1]), "failed": ";".join(failed)}


def score_compound(run, compound, smiles):
    pred = find_pred_dir(run, compound)
    if pred is None:
        return []
    rows = []
    for pdb in sorted(pred.glob(f"{compound}_model_*.pdb")):
        conf = json.loads((pred / f"confidence_{pdb.stem}.json").read_text())["confidence_score"]
        res = posebusters(pdb, smiles)
        rows.append({"compound": compound, "sample": pdb.stem.split("_")[-1], "confidence": round(conf, 4), "pdb": str(pdb), **res})
    return rows


def score_run(run, frame, conn=None):
    """Run PoseBusters on every finished compound. Safe to rerun. If conn is given (the results database,
    section 18), each compound's rows are written there too, right after the checkpoint CSV."""
    out_csv = ROOT / "runs" / run / "posebusters_samples.csv"
    have = pd.read_csv(out_csv) if out_csv.exists() else pd.DataFrame()
    done = set(have["compound"]) if len(have) else set()
    heavy = dict(zip(frame["Molecule_Name"], frame["heavy_atoms"])) if "heavy_atoms" in frame else {}
    for name, smi in tqdm(list(zip(frame["Molecule_Name"], frame["SMILES"]))):
        if name in done:
            continue
        rows = score_compound(run, name, smi)
        if rows:
            have = pd.concat([have, pd.DataFrame(rows)], ignore_index=True)
            have.to_csv(out_csv, index=False)
            if conn is not None:
                db_write_compound_results(conn, run, name, smi, heavy.get(name, 0), "boltz", rows)
    return have


# Uncomment to score all finished compounds (about 1 to 2 s per sample).
# samples_df = score_run(RUN, df)
# samples_df.groupby("compound").apply(lambda g: (g.n_pass == g.n_total).sum()).value_counts().sort_index()

## 8. Where does a failing pose clash
Lists the closest ligand to protein heavy-atom pairs in one PDB. A hydrogen-bonding N or O pair below about 2.3 A, or a carbon pair below about 3.0 A, is a likely clash.
Compound 001 fails `minimum_distance_to_protein` in all five samples, and this shows which atoms are involved.

In [ ]:
def read_atoms(pdb_path):
    rows = []
    for l in open(pdb_path):
        if l.startswith(("ATOM", "HETATM")):
            elem = l[76:78].strip() or l[12:16].strip()[0]
            if elem == "H":
                continue
            rows.append((l[12:16].strip(), l[17:20].strip(), l[22:26].strip(),
                         np.array([float(l[30:38]), float(l[38:46]), float(l[46:54])])))
    return rows


def closest_contacts(pdb_path, n=5):
    atoms = read_atoms(pdb_path)
    lig = [a for a in atoms if a[1] == "LIG"]
    prot = [a for a in atoms if a[1] != "LIG"]
    L = np.array([a[3] for a in lig])
    P = np.array([a[3] for a in prot])
    D = np.linalg.norm(L[:, None] - P[None], axis=2)
    idx = np.dstack(np.unravel_index(np.argsort(D, axis=None)[:n], D.shape))[0]
    return pd.DataFrame([{"ligand_atom": lig[i][0], "protein": f"{prot[j][1]}{prot[j][2]} {prot[j][0]}", "distance_A": round(float(D[i, j]), 2)} for i, j in idx])


# Example, uncomment after the pilot run exists:
# closest_contacts(next((ROOT / "runs/pilot/out").rglob("blind_challenge_001_model_0.pdb")))

## 9. Pick one pose per compound
The challenge needs exactly one pose per compound. The rule for now follows the organizers' baseline.
Keep samples that pass all checks, then take the one with the highest Boltz confidence.
If no sample passes, take the one with the most checks passed and flag it as not valid.
Later this cell is where the smarter selector (sample agreement, templates, relaxation) goes.

In [ ]:
def pick_best(samples):
    rows = []
    for comp, g in samples.groupby("compound"):
        g = g.assign(valid=g["n_pass"] == g["n_total"])
        g = g.sort_values(["valid", "n_pass", "confidence"], ascending=False)
        best = g.iloc[0]
        rows.append({"compound": comp, "sample": best["sample"], "valid": bool(best["valid"]),
                     "n_pass": int(best["n_pass"]), "failed": best["failed"], "confidence": best["confidence"],
                     "n_valid_samples": int(g["valid"].sum()), "pdb": best["pdb"]})
    return pd.DataFrame(rows)


# Quick self test of the rule with fake data.
_t = pd.DataFrame([
    {"compound": "a", "sample": "0", "confidence": 0.95, "n_pass": 21, "n_total": 22, "failed": "x", "pdb": "a0"},
    {"compound": "a", "sample": "1", "confidence": 0.94, "n_pass": 22, "n_total": 22, "failed": "", "pdb": "a1"},
    {"compound": "b", "sample": "0", "confidence": 0.90, "n_pass": 21, "n_total": 22, "failed": "x", "pdb": "b0"},
    {"compound": "b", "sample": "1", "confidence": 0.93, "n_pass": 21, "n_total": 22, "failed": "x", "pdb": "b1"},
])
_p = pick_best(_t).set_index("compound")
assert _p.loc["a", "pdb"] == "a1" and _p.loc["a", "valid"]
assert _p.loc["b", "pdb"] == "b1" and not _p.loc["b", "valid"]
print("selection rule OK")

# Uncomment after score_run has finished:
# selected = pick_best(samples_df)
# print(selected["valid"].value_counts())
# selected.to_csv(ROOT / "runs" / RUN / "selected.csv", index=False)

## 10. Official scorer and its self test
The organizers score with OpenStructure, which lives in `zika-eval`. This cell writes a tiny helper script to the scratch folder and runs it there.
The self test checks the scorer is trustworthy. A pose compared with itself must give BiSyRMSD near 0 and lDDT-PLI near 1, and a ligand shifted by 2 A must give BiSyRMSD of 2.
We only have reference structures for public complexes, so this is for local benchmarking, not for the challenge compounds.

In [ ]:
SCORER_PY = r'''
import sys, json
sys.path.insert(0, sys.argv[3])
from evaluation.evaluate_predictions import score_single_structure
res = score_single_structure(sys.argv[1], sys.argv[2])
print("RESULT " + json.dumps({k: float(v) for k, v in res.items()}))
'''


def score_vs_reference(model_pdb, ref_pdb):
    script = SCRATCH / "score_one.py"
    script.write_text(SCORER_PY)
    r = run_in_env("zika-eval", ["python", str(script), str(model_pdb), str(ref_pdb), str(TUTORIAL)], cwd=TUTORIAL)
    for line in r.stdout.splitlines():
        if line.startswith("RESULT "):
            return json.loads(line[7:])
    raise RuntimeError(r.stderr[-1500:] or r.stdout[-1500:])


def scorer_self_test():
    ref = TUTORIAL / "outputs/my_structure_submission/blind_challenge_001.pdb"
    same = score_vs_reference(ref, ref)
    shifted = SCRATCH / "shifted.pdb"
    lines = []
    for l in ref.read_text().splitlines(True):
        if l.startswith(("ATOM", "HETATM")) and l[17:20].strip() == "LIG":
            l = l[:30] + f"{float(l[30:38]) + 2.0:8.3f}" + l[38:]
        lines.append(l)
    shifted.write_text("".join(lines))
    moved = score_vs_reference(shifted, ref)
    assert same["BiSyRMSD"] < 0.01 and same["LDDT-PLI"] > 0.99, same
    assert abs(moved["BiSyRMSD"] - 2.0) < 0.05, moved
    print("scorer OK | same:", same, "| shifted 2 A:", moved)


# Uncomment to run (takes under a minute).
# scorer_self_test()

## 11. Clash relaxation with OpenMM (to be added)
Compounds like 001 fail one PoseBusters check in every sample, so more samples will not fix them. The plan is to relax the protein pocket around the fixed ligand in OpenMM (env `zika-md`, CPU only) and rerun PoseBusters.
A relaxed pose only counts if the ligand moved less than 0.5 A and the backbone less than 0.3 A, and the poses that already pass must still pass.
The function goes here once the Claude Code session returns it. It will be called through `run_in_env("zika-md", ...)`.
Known issue. OpenMM CUDA fails on the laptop driver (PTX version error), so use the CPU platform.

## 11a. Ligand atom cleanup (shared by scoring and the submission writer)
Testing the pose-agreement scorer input (section 19) surfaced a real bug: GNINA's complex PDBs name every ligand atom by its bare element symbol, so several atoms end up named plain `C`, plain `N`, and so on. PDB atom names must be unique within a residue. OpenStructure's own reader keeps only the first atom with each repeated name and silently drops the rest, which once shrank a 21-heavy-atom ligand down to 3 or 4 atoms with no error raised, so the organizers' scorer would have scored a broken ligand as a near-total pose failure. GNINA's poses also carry explicit ligand hydrogens, which Boltz's output never has.

`clean_ligand_lines` is the single fix for both: it drops ligand hydrogens and any non-primary alternate location, and renames every kept atom to a unique element-plus-index name (Boltz's own convention, for example `C1`, `C2`, `N1`). It is used both by `format_submission_pdb` below (so no file that goes into the submission ZIP can have this problem, whichever method made the pose) and by `materialize_for_scoring` (section 19, refactored to call it too), so every consumer of a pose file sees the same fix instead of each keeping its own copy.

In [ ]:
from collections import defaultdict


def is_hydrogen(atom_line):
    """True if one ATOM or HETATM line is a hydrogen (or deuterium), read from the element column
    (cols 77-78) first and the atom name only as a fallback."""
    elem = atom_line[76:78].strip() or atom_line[12:16].strip()[:1]
    return elem.upper() in ("H", "D")


def ligand_element(atom_line):
    """Element symbol for one ligand ATOM or HETATM line: the element column (cols 77-78) if it is
    set, otherwise the alphabetic prefix of the atom name, so a two-letter element such as Cl is not
    truncated to C."""
    elem = atom_line[76:78].strip()
    if elem:
        return elem[:1].upper() + elem[1:].lower()
    letters = "".join(ch for ch in atom_line[12:16].strip() if ch.isalpha())
    return letters[:1].upper() + letters[1:2].lower()


def clean_ligand_lines(lig_lines):
    """Build a canonical heavy-atom-only ligand block from one pose's raw LIG atom lines: drop
    ligand hydrogens and any non-primary alternate location, rename every kept atom to a unique
    element-plus-index name (Boltz's own convention), and put every atom on chain C, residue number
    1. Every kept atom keeps its original coordinate, occupancy, and temperature columns unchanged;
    only the name, altLoc, chain, and residue-number columns change."""
    kept = [l for l in lig_lines if l[16] in (" ", "A") and not is_hydrogen(l)]
    ligand_tail = " " + "LIG" + " " + "C" + f"{1:>4}" + " " * 4
    counts = defaultdict(int)
    cleaned = []
    for l in kept:
        elem = ligand_element(l)
        counts[elem] += 1
        name = f"{elem}{counts[elem]}"
        name_field = f" {name:<3}"[:4]
        cleaned.append(f"{l[:12]}{name_field}{ligand_tail}{l[30:]}")
    return cleaned


def _blank_altloc(line):
    """Blank the alternate-location flag (column 17) of one PDB ATOM or HETATM line."""
    return line[:16] + " " + line[17:]


def _set_serial(line, serial, record):
    """Rewrite the record type and atom serial number of one PDB line. Every other column,
    including the coordinates, is copied through unchanged."""
    return f"{record:<6}{serial:>5}" + line[11:]


### Submission-format writer for one pose
`format_submission_pdb` is the single place that turns any pose (Boltz, the Boltz 29HN-template variant, or GNINA) into the exact submission format, so no file that goes into the submission ZIP can have the duplicate-atom-name problem above, whichever method made the pose. It changes no atom's coordinates; it only rewrites names, chain/residue identifiers, altLoc flags, and serial numbers.

In [ ]:
def format_submission_pdb(src_pdb, out_pdb):
    """Rewrite one predicted complex into the exact submission format, regardless of which method
    produced it:

    - protein chains A and B, every kept atom unchanged except for a blanked altLoc flag
    - ligand chain C, residue name LIG, exactly one LIG residue, heavy atoms only
    - unique ligand atom names built from element plus index (clean_ligand_lines, section 11a)
    - no alternate-location letters anywhere in the file
    - no duplicate atom serials (the whole file is renumbered from 1)
    - no NaN coordinates

    No atom coordinate is changed for any kept atom. Raises ValueError if the source file cannot be
    made to fit this format (missing protein chain, more than one ligand residue, no heavy ligand
    atoms, or a NaN coordinate), rather than silently writing a bad file.
    """
    src_pdb = Path(src_pdb)
    lines = src_pdb.read_text().splitlines(True)
    atom_lines = [l for l in lines if l.startswith(("ATOM", "HETATM"))]
    lig_lines = [l for l in atom_lines if l[17:20].strip() == "LIG"]
    protein_lines = [l for l in atom_lines if l[17:20].strip() != "LIG"]

    lig_residues = {(l[21], l[22:26]) for l in lig_lines}
    if len(lig_residues) > 1:
        raise ValueError(f"{src_pdb}: ligand atoms span {len(lig_residues)} residues, expected 1")

    protein_lines = [_blank_altloc(l) for l in protein_lines if l[16] in (" ", "A")]
    protein_chains = {l[21] for l in protein_lines}
    if not {"A", "B"} <= protein_chains:
        raise ValueError(f"{src_pdb}: protein chains found {sorted(protein_chains)}, expected A and B")

    cleaned_lig = clean_ligand_lines(lig_lines)
    if not cleaned_lig:
        raise ValueError(f"{src_pdb}: no heavy ligand atoms found")

    for l in protein_lines + cleaned_lig:
        x, y, z = float(l[30:38]), float(l[38:46]), float(l[46:54])
        if x != x or y != y or z != z:
            raise ValueError(f"{src_pdb}: NaN coordinate in line {l!r}")

    out_lines = [_set_serial(l, i + 1, "ATOM") for i, l in enumerate(protein_lines)]
    out_lines += [_set_serial(l, len(protein_lines) + i + 1, "HETATM") for i, l in enumerate(cleaned_lig)]
    out_pdb = Path(out_pdb)
    out_pdb.parent.mkdir(parents=True, exist_ok=True)
    out_pdb.write_text("".join(out_lines) + "END\n")
    return out_pdb


## 12. Build the submission ZIP
Writes one submission-format PDB per compound with `format_submission_pdb` (section 11b), checks the basics, zips them, and runs the organizers' validator.
First look at how the organizers' example ZIP is laid out, so ours matches.

In [ ]:
example_zip = TUTORIAL / "outputs" / "example_structure_submission.zip"
names = zipfile.ZipFile(example_zip).namelist()
print(len(names), "files in the example zip")
print(names[:3])

### Official validator
The organizers' validator needs MDAnalysis, which lives in `zika-eval`, so `validate_zip` runs it there. It checks the ZIP, that every expected ID is present, that each file has exactly one `LIG` residue and at most three chains, and that the ligand matches its SMILES. It does not check the protein chains, so `check_submission` below does.
`expected_ids` and `expected_ligand_smiles` are always built from the `frame` argument (default the full 400-compound `df`), never hardcoded to all 400, so a partial-compound ZIP (for example a 5-compound test run) validates against only the compounds it is supposed to contain, instead of being flagged for every compound it was never meant to hold.
The last line is a self test on the organizers' example ZIP. Expect `(True, [])`.

In [ ]:
VALIDATOR_PY = r'''
import sys, json
import pandas as pd
sys.path.insert(0, sys.argv[2])
from validation.structure_validation import validate_structure_submission
df = pd.read_csv(sys.argv[3])
ok, errors = validate_structure_submission(sys.argv[1], expected_ids=set(df["Molecule_Name"]),
                                           expected_ligand_smiles=dict(zip(df["Molecule_Name"], df["SMILES"])))
print("RESULT " + json.dumps({"ok": ok, "errors": errors}))
'''


def validate_zip(zip_path, frame=None):
    """Run the organizers' validator on a ZIP. expected_ids and expected_ligand_smiles are always
    built from frame (defaults to the full challenge set, df), so a partial-compound ZIP (for
    example a 5-compound test run) validates cleanly against only the compounds it is supposed to
    contain, rather than being flagged for every one of the other compounds it was never meant to
    hold."""
    frame = df if frame is None else frame
    script = SCRATCH / "validate_zip.py"
    script.write_text(VALIDATOR_PY)
    frame_csv = SCRATCH / "validate_zip_compounds.csv"
    frame[["Molecule_Name", "SMILES"]].to_csv(frame_csv, index=False)
    r = run_in_env("zika-eval", ["python", str(script), str(zip_path), str(TUTORIAL), str(frame_csv)], cwd=TUTORIAL)
    for line in r.stdout.splitlines():
        if line.startswith("RESULT "):
            res = json.loads(line[7:])
            return res["ok"], res["errors"]
    raise RuntimeError(r.stderr[-1500:] or r.stdout[-1500:])


# Self test on the organizers' example zip. Uncomment to run.
# validate_zip(example_zip, df)


### Build, check, and zip
`build_submission` writes the chosen pose for each compound through `format_submission_pdb`, so every file in the submission is already in the exact submission format, whichever method produced it. `check_one_submission_file` is the per-file half of our own checks (the official validator does not test the protein chains, the ligand atom names, or the SMILES heavy-atom count); `check_submission` runs it over a finished submission folder plus the file-count and missing-compound checks. `make_zip` writes a dated ZIP.
The batch readability check (OpenStructure's own reader) is its own subsection right after this one.
The last cell is commented out until `selected` covers all 400 compounds. Run the official validator and the readability check on the ZIP before uploading.

In [ ]:
def build_submission(selected, run):
    """Write one submission-format PDB per compound (format_submission_pdb, section 11b), from
    whichever method's pose was selected. Works the same for a Boltz, Boltz-template, or GNINA
    pose, since the writer only looks at the PDB records, not the source."""
    sub = ROOT / "runs" / run / "submission"
    sub.mkdir(parents=True, exist_ok=True)
    for _, r in selected.iterrows():
        format_submission_pdb(r["pdb"], sub / f"{r['compound']}.pdb")
    return sub


def check_one_submission_file(pdb_path, expected_heavy_atoms):
    """Per-file checks for one PDB that is supposed to already be in submission format: exactly one
    LIG residue, protein chains A and B present, ligand atom names unique, ligand heavy-atom count
    equal to expected_heavy_atoms (the compound's RDKit SMILES heavy-atom count), and no NaN
    coordinates. Returns a list of problem strings, empty if the file is clean."""
    pdb_path = Path(pdb_path)
    atoms = [l for l in pdb_path.read_text().splitlines() if l.startswith(("ATOM", "HETATM"))]
    lig = [l for l in atoms if l[17:20].strip() == "LIG"]
    protein = [l for l in atoms if l[17:20].strip() != "LIG"]
    chains = {l[21] for l in protein}
    lig_residues = {(l[21], l[22:26]) for l in lig}
    names = [l[12:16].strip() for l in lig]

    problems = []
    if not lig:
        problems.append(f"{pdb_path.name} has no LIG residue")
    elif len(lig_residues) != 1:
        problems.append(f"{pdb_path.name} has {len(lig_residues)} LIG residues, expected 1")
    if not {"A", "B"} <= chains:
        problems.append(f"{pdb_path.name} protein chains found {sorted(chains)}, expected A and B")
    if len(set(names)) != len(names):
        problems.append(f"{pdb_path.name} has duplicate ligand atom names")
    if expected_heavy_atoms is not None and len(lig) != expected_heavy_atoms:
        problems.append(f"{pdb_path.name} has {len(lig)} ligand atoms, expected {expected_heavy_atoms} (SMILES heavy atoms)")
    for l in atoms:
        x, y, z = float(l[30:38]), float(l[38:46]), float(l[46:54])
        if x != x or y != y or z != z:
            problems.append(f"{pdb_path.name} has a NaN coordinate")
            break
    return problems


def check_submission(sub, frame):
    """Our own checks, run on top of the official validator (which does not test the protein
    chains, the ligand atom names, or the SMILES heavy-atom count): the right number of files for
    frame, none missing, and check_one_submission_file on every file."""
    files = sorted(Path(sub).glob("blind_challenge_*.pdb"))
    heavy_atoms = dict(zip(frame["Molecule_Name"], frame["heavy_atoms"]))
    problems = []
    if len(files) != len(frame):
        problems.append(f"{len(files)} files, expected {len(frame)}")
    missing = set(frame["Molecule_Name"]) - {f.stem for f in files}
    if missing:
        problems.append(f"missing: {sorted(missing)[:5]} ...")
    for f in files:
        problems += check_one_submission_file(f, heavy_atoms.get(f.stem))
    return problems


def make_zip(sub, run):
    """Zip every PDB in a finished submission folder, dated so repeated attempts do not collide."""
    name = ROOT / "runs" / run / f"submission_{datetime.date.today():%Y%m%d}.zip"
    with zipfile.ZipFile(name, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(Path(sub).glob("*.pdb")):
            z.write(f, f.name)
    return name


# Uncomment when selected.csv covers all 400 compounds.
# sub = build_submission(selected, RUN)
# print(check_submission(sub, df) or "basic checks passed")
# print(check_submission_readability(sub, df)[1] or "readability checks passed")
# zip_path = make_zip(sub, RUN)
# print(zip_path)
# print(validate_zip(zip_path, df))


### Batch readability check (OpenStructure)
`format_submission_pdb` and `check_one_submission_file` work on the PDB text, but the organizers score with OpenStructure, and OpenStructure's own PDB reader is a separate code path with its own ideas about what counts as a valid atom (this is exactly how the duplicate-atom-name bug in section 11a was first found: a file could look fine on paper and still read back wrong). `check_readability` loads every file with that same reader, in one `zika-eval` call like the other scorer helpers, and returns the ligand atom count it actually sees per file. `check_submission_readability` is the convenience wrapper for a finished submission folder: it must equal the SMILES heavy-atom count for every file, and flags any file where it does not.

In [ ]:
READABILITY_PY = r'''
import sys, json
from ost import io

files = json.loads(open(sys.argv[1]).read())
results = []
for item in files:
    try:
        entity = io.LoadPDB(item["path"], fault_tolerant=True)
        n = entity.Select("rname=LIG").GetAtomCount()
        results.append({"key": item["key"], "n_ligand_atoms": n, "error": None})
    except Exception as e:
        results.append({"key": item["key"], "n_ligand_atoms": None, "error": str(e)})
json.dump(results, open(sys.argv[2], "w"))
'''


def check_readability(files):
    """Load every file in `files` (a list of (key, path) pairs) with OpenStructure's own PDB reader
    (ost.io.LoadPDB), in one zika-eval call, and return the ligand atom count OpenStructure sees for
    each key."""
    script = SCRATCH / "check_readability.py"
    script.write_text(READABILITY_PY)
    in_json, out_json = SCRATCH / "readability_in.json", SCRATCH / "readability_out.json"
    in_json.write_text(json.dumps([{"key": key, "path": str(path)} for key, path in files]))
    if out_json.exists():
        out_json.unlink()
    r = run_in_env("zika-eval", ["python", str(script), str(in_json), str(out_json)])
    if not out_json.exists():
        raise RuntimeError(r.stderr[-2000:] or r.stdout[-2000:])
    return pd.DataFrame(json.loads(out_json.read_text()))


def mismatched_ligand_counts(files, expected):
    """Run check_readability on files ((key, path) pairs) and flag any key whose OpenStructure
    ligand atom count does not equal expected[key] (the compound's SMILES heavy-atom count).
    Returns (full_result, mismatches_only)."""
    result = check_readability(files)
    result["expected"] = result["key"].map(expected)
    result["ok"] = result["n_ligand_atoms"] == result["expected"]
    return result, result[~result["ok"]]


def check_submission_readability(sub, frame):
    """Batch readability check for a finished submission folder: every blind_challenge_*.pdb must
    load with OpenStructure and show a ligand atom count equal to that compound's SMILES
    heavy-atom count. Returns (full_result, mismatches_only)."""
    heavy_atoms = dict(zip(frame["Molecule_Name"], frame["heavy_atoms"]))
    files = [(f.stem, f) for f in sorted(Path(sub).glob("blind_challenge_*.pdb"))]
    return mismatched_ligand_counts(files, heavy_atoms)


## 13. Decisions and findings so far
A running record, which also feeds the methods report.

- Boltz-2 on the RTX 4070 laptop takes about 43 s per compound for 5 samples (3-compound pilot).
- WSL was capped at 6 GB and a 3-compound run was killed after its results were written. The cap is now 10 GB in `.wslconfig` and applies after `wsl --shutdown`.
- PoseBusters on the pilot. Compound 002 had 4 of 5 valid samples, 003 had 2 of 5, and 001 had 0 of 5.
- All five 001 samples fail `minimum_distance_to_protein`. The tutorial's own example pose for 001 fails the same check (ligand N to the Tyr115 backbone O at 2.18 A).
- The organizers' Boltz baseline reached 88.25 percent of compounds with a full 22-check pass by picking among 5 samples.
- fpocket ranked the true binding site third on a Boltz model, so the site is defined from template ligands, not pocket rank.
- The scorer self test passed. Identical pose gave BiSyRMSD near 0, and a 2 A shift gave 2.0 with lDDT-PLI 0.69.
- Package pins in `zika`. Boltz needs `click==8.1.7` and numpy 1.26.4. OpenMM and OpenStructure live in their own envs because they want numpy 2.x.
- The tutorial YAML for compound 001 merged NS2B and NS3 into one chain. Our builder reads the FASTA instead.
- Dry run `test5` (5 compounds, SAMPLES=2). Boltz baseline matched the pilot: 001 fails `minimum_distance_to_protein` again, 002-005 pass. GNINA docking passed all 22 PoseBusters checks on all 10 poses (5 compounds x 2 samples), including compound 001, so the database's per-compound best-pose query picks GNINA for 001 and Boltz for the rest.
- 29HN has alternate locations on a few residues; `convert_receptor` keeps only the first altloc and blanks the altloc column, otherwise Meeko's receptor prep refuses the file.
- Boltz's PDB template parser (`parse_pdb` -> `parse_mmcif` -> `parse_polymer`) raised `IndexError: list index out of range` on our stripped protein-only PDB (no SEQRES/entity_poly records for gemmi to align against). Using the original `29HN.cif` as the template fixed it; verified with a one-compound `boltz predict` (0 failed examples), then the full 5-compound template batch (0 failed examples).
- GNINA v1.1 runs on the GPU here, about 18x faster than `--no_gpu` CPU fallback on one probe ligand (6 s vs 110 s).
- Boltz confidence_score and GNINA's CNNscore are not calibrated against each other, they are just both roughly 0-1 "higher is better" scores from different models. The database's cross-method pose selection treats them as comparable; treat that as a simplification, not a validated equivalence.

## 14. Receptor preparation for docking (29HN)
29HN is the solved apo NS2B-NS3 structure (chain A is NS2B, chain B is NS3). It already exists in `structures/`, so it is picked with the same `PathPicker` used everywhere else, not downloaded.
`convert_receptor` keeps only the two protein chains and drops waters and any other heteroatoms, but keeps the author residue numbering (`auth_seq_id`), because that is the numbering the paper and this notebook use for the key residues (NS3 His51, Asp75, Ser135 are the catalytic triad).
`verify_catalytic_triad` checks those three residues landed where expected, and `summarize_receptor` prints chains, residue counts, and the residue number range.

In [ ]:
_receptor_p = PathPicker("Receptor structure (29HN.cif)", filt="mmCIF files|*.cif|All files|*.*",
                         root_fn=_project, default=_auto("29HN.cif"))
_receptor_status = w.Output()


def set_receptor_path(_=None):
    """Read the receptor picker and check the file exists."""
    global RECEPTOR_CIF
    RECEPTOR_CIF = _receptor_p.path
    with _receptor_status:
        _receptor_status.clear_output()
        ok = RECEPTOR_CIF is not None and RECEPTOR_CIF.exists()
        print(f"{'OK     ' if ok else 'MISSING'} Receptor structure: {RECEPTOR_CIF}")


_receptor_confirm = w.Button(description="Confirm receptor path", button_style="primary")
_receptor_confirm.on_click(set_receptor_path)
display(w.VBox([_receptor_p.widget, _receptor_confirm, _receptor_status]))
set_receptor_path()

### Convert and verify
The CIF has alternate locations for a few residues (disordered side chains). `convert_receptor` keeps only the first altloc for those atoms and blanks the altloc flag in the output, otherwise Meeko's receptor prep refuses the file later. The output is saved next to the CIF as `29HN_protein.pdb`.

In [ ]:
from Bio.PDB import MMCIFParser, PDBParser, PDBIO, Select

PROTEIN_CHAINS = ("A", "B")
# NS3's catalytic triad, in the paper's author numbering (chain B residues).
CATALYTIC_TRIAD = [("B", 51, "HIS"), ("B", 75, "ASP"), ("B", 135, "SER")]


class _ProteinOnly(Select):
    """Keeps standard residues (no waters, no other heteroatoms) in the given chains, one altloc per atom."""

    def __init__(self, chains):
        self.chains = set(chains)

    def accept_chain(self, chain):
        return chain.id in self.chains

    def accept_residue(self, residue):
        return residue.id[0] == " " and residue.get_parent().id in self.chains

    def accept_atom(self, atom):
        return atom.get_altloc() in (" ", "A")


def convert_receptor(cif_path, out_pdb, chains=PROTEIN_CHAINS):
    """Convert a solved structure CIF to a clean protein-only PDB (no waters), keeping author numbering."""
    parser = MMCIFParser(QUIET=True, auth_chains=True, auth_residues=True)
    structure = parser.get_structure("receptor", str(cif_path))
    io = PDBIO()
    io.set_structure(structure)
    io.save(str(out_pdb), select=_ProteinOnly(chains))
    # Blank the altloc column so downstream tools (Meeko) see one unambiguous conformer.
    lines = Path(out_pdb).read_text().splitlines(True)
    lines = [l[:16] + " " + l[17:] if l.startswith(("ATOM", "HETATM")) else l for l in lines]
    Path(out_pdb).write_text("".join(lines))
    return Path(out_pdb)


def summarize_receptor(pdb_path):
    """Print chains, residue counts, and the residue number range in a converted receptor PDB."""
    structure = PDBParser(QUIET=True).get_structure("receptor", str(pdb_path))
    rows = []
    for chain in structure[0]:
        nums = [res.id[1] for res in chain]
        rows.append({"chain": chain.id, "residues": len(nums), "range": f"{min(nums)}-{max(nums)}"})
    table = pd.DataFrame(rows)
    print(table.to_string(index=False))
    return table


def verify_catalytic_triad(pdb_path, triad=CATALYTIC_TRIAD):
    """Check that the key residues exist at the expected author-numbered positions."""
    structure = PDBParser(QUIET=True).get_structure("receptor", str(pdb_path))
    model = structure[0]
    rows = []
    for chain_id, resnum, expect in triad:
        res = model[chain_id].child_dict.get((" ", resnum, " "))
        rows.append({"chain": chain_id, "resnum": resnum, "expected": expect,
                     "found": res.resname if res else "MISSING", "ok": bool(res) and res.resname == expect})
    table = pd.DataFrame(rows)
    assert table["ok"].all(), f"catalytic triad check failed:\n{table}"
    print(table.to_string(index=False))
    return table


RECEPTOR_PDB = RECEPTOR_CIF.parent / "29HN_protein.pdb"
convert_receptor(RECEPTOR_CIF, RECEPTOR_PDB)
print("receptor written to", RECEPTOR_PDB)
summarize_receptor(RECEPTOR_PDB)
verify_catalytic_triad(RECEPTOR_PDB)

## 15. GNINA docking setup: receptor PDBQT and the docking box
`prepare_receptor_pdbqt` converts the clean protein PDB to a rigid PDBQT with Meeko (`mk_prepare_receptor.py`).
`compute_docking_box` builds the search box from a parameter list of key residues plus a padding value, rather than a fixed guess: it takes every atom of those residues and wraps a padded box around them. Edit the list or the padding below and click Apply.
Defaults: NS3 His51, Asp129, Tyr130, Ser135, Tyr150, Tyr161, and NS2B Asp83 (the catalytic triad plus the substrate-pocket residues used to define the binding cleft), with 5 A padding.

In [ ]:
def prepare_receptor_pdbqt(receptor_pdb, out_pdbqt):
    """Convert the clean protein PDB into a rigid PDBQT receptor with Meeko. Meeko's template adds standard
    polar hydrogens here; it does not know about the crystal's pH 4.8, see the note in section 16."""
    r = subprocess.run(["mk_prepare_receptor.py", "--read_pdb", str(receptor_pdb),
                        "-o", str(Path(out_pdbqt).with_suffix("")), "-p"], capture_output=True, text=True)
    if r.returncode != 0 or not Path(out_pdbqt).exists():
        raise RuntimeError(r.stdout[-1500:] + r.stderr[-1500:])
    return Path(out_pdbqt)


RECEPTOR_PDBQT = RECEPTOR_PDB.with_suffix(".pdbqt")
prepare_receptor_pdbqt(RECEPTOR_PDB, RECEPTOR_PDBQT)
print("receptor PDBQT:", RECEPTOR_PDBQT)


def compute_docking_box(pdb_path, residues, padding):
    """Build an axis-aligned box covering every atom of the given (chain, resnum, resname) residues, plus
    padding (Angstrom) on every side."""
    wanted = {(c, n) for c, n, _ in residues}
    present = {(l[21], int(l[22:26])) for l in open(pdb_path) if l.startswith("ATOM")}
    missing = wanted - present
    if missing:
        raise ValueError(f"key residues not found in {pdb_path}: {sorted(missing)}")
    coords = np.array([[float(l[30:38]), float(l[38:46]), float(l[46:54])]
                       for l in open(pdb_path) if l.startswith("ATOM") and (l[21], int(l[22:26])) in wanted])
    lo, hi = coords.min(0), coords.max(0)
    center, size = (lo + hi) / 2, (hi - lo) + 2 * padding
    return {"center_x": round(float(center[0]), 3), "center_y": round(float(center[1]), 3), "center_z": round(float(center[2]), 3),
            "size_x": round(float(size[0]), 3), "size_y": round(float(size[1]), 3), "size_z": round(float(size[2]), 3)}


def _parse_residues(text, pdb_path):
    """Parse 'B51, A83, ...' into (chain, resnum, resname), reading the resname straight from the receptor
    so a typo'd residue number fails loudly instead of silently picking the wrong atom."""
    names = {}
    for l in open(pdb_path):
        if l.startswith("ATOM"):
            names[(l[21], int(l[22:26]))] = l[17:20].strip()
    out = []
    for tok in text.split(","):
        tok = tok.strip()
        if not tok:
            continue
        chain, resnum = tok[0], int(tok[1:])
        resname = names.get((chain, resnum))
        if resname is None:
            raise ValueError(f"{tok} not found in {pdb_path}")
        out.append((chain, resnum, resname))
    return out


KEY_RESIDUES = [("B", 51, "HIS"), ("B", 129, "ASP"), ("B", 130, "TYR"), ("B", 135, "SER"),
                ("B", 150, "TYR"), ("B", 161, "TYR"), ("A", 83, "ASP")]
BOX_PADDING = 5.0

_res_text = w.Textarea(value=", ".join(f"{c}{n}" for c, n, _ in KEY_RESIDUES),
                       description="key residues", placeholder="e.g. B51, B129, A83",
                       style=_pstyle, layout=w.Layout(width="640px", height="50px"))
_pad_box = w.FloatText(value=BOX_PADDING, description="padding (A)", style=_pstyle, layout=_lay())
_box_apply = w.Button(description="Apply box parameters", button_style="primary")
_box_status = w.Output()


def apply_box_params(_=None):
    global KEY_RESIDUES, BOX_PADDING, DOCKING_BOX
    with _box_status:
        _box_status.clear_output()
        KEY_RESIDUES = _parse_residues(_res_text.value, RECEPTOR_PDB)
        BOX_PADDING = _pad_box.value
        DOCKING_BOX = compute_docking_box(RECEPTOR_PDB, KEY_RESIDUES, BOX_PADDING)
        print("key residues:", ", ".join(f"{c}{n}({r})" for c, n, r in KEY_RESIDUES))
        print("padding:", BOX_PADDING, "A")
        print("box:", DOCKING_BOX)


_box_apply.on_click(apply_box_params)
display(w.VBox([_res_text, _pad_box, _box_apply, _box_status]))
apply_box_params()

### Ligand preparation
`prepare_ligand` embeds one 3D conformer from the CSV SMILES exactly as given (no atoms added, removed, or re-perceived), optimizes it, and converts it to PDBQT with Meeko. The prep record saved alongside each compound says which embedding seed and force field were used.

**Open question, not decided here.** 29HN was crystallized at pH 4.8. Neither the ligand (RDKit's default valence from the SMILES) nor the receptor (Meeko's standard-residue protonation in the PDBQT step above) has its protonation state adjusted for that pH. His51, Asp75, Asp129, and the ligands' own ionizable groups could plausibly sit in different protonation states at pH 4.8 than the defaults used here. This needs a decision (for example a pH-aware tool such as PDB2PQR, or explicit manual states) before the docking scores can be read quantitatively.

In [ ]:
def prepare_ligand(name, smiles, out_dir, seed=SEED):
    """Build one 3D conformer from the CSV SMILES exactly as given, and convert it to PDBQT with Meeko.
    Returns the PDBQT path and a record of how the ligand was prepared (see the pH note above)."""
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    mol = Chem.AddHs(Chem.MolFromSmiles(smiles))
    force_field = "MMFF94"
    if AllChem.EmbedMolecule(mol, randomSeed=seed) < 0:
        AllChem.EmbedMolecule(mol, randomSeed=seed, useRandomCoords=True)
    if AllChem.MMFFOptimizeMolecule(mol) != 0:
        AllChem.UFFOptimizeMolecule(mol)
        force_field = "UFF (MMFF94 failed)"
    sdf_path = out_dir / f"{name}.sdf"
    writer = Chem.SDWriter(str(sdf_path))
    writer.write(mol)
    writer.close()
    pdbqt_path = out_dir / f"{name}.pdbqt"
    r = subprocess.run(["mk_prepare_ligand.py", "-i", str(sdf_path), "-o", str(pdbqt_path)],
                       capture_output=True, text=True)
    if r.returncode != 0 or not pdbqt_path.exists():
        raise RuntimeError(r.stderr[-1500:] or r.stdout[-1500:])
    record = {"compound": name, "smiles": smiles, "embed_seed": seed, "force_field": force_field,
              "protonation": "RDKit AddHs on the exact CSV SMILES, no pH-based protonation state chosen"}
    return pdbqt_path, record

### GPU check
`gnina_gpu_check` docks one probe ligand (the first compound) to see whether GNINA's GPU path works on this machine, timing it. If the GPU run fails, it retries with `--no_gpu` (CPU). If that also fails, it falls back to plain AutoDock Vina. Whichever one works becomes `DOCK_BACKEND`, used for every compound below.

In [ ]:
import time

GNINA_BIN = Path.home() / "gnina"


def _box_args(box):
    return ["--center_x", str(box["center_x"]), "--center_y", str(box["center_y"]), "--center_z", str(box["center_z"]),
            "--size_x", str(box["size_x"]), "--size_y", str(box["size_y"]), "--size_z", str(box["size_z"])]


def gnina_gpu_check(receptor_pdbqt, ligand_pdbqt, box, timeout=120):
    """Try GNINA on the GPU first. Falls back to GNINA on the CPU, then to plain Vina, and reports which one
    worked plus the time taken. Returns (backend, seconds), backend is 'gpu', 'cpu', or 'vina'."""
    def _probe(args, timeout):
        t0 = time.time()
        try:
            r = subprocess.run(args, capture_output=True, text=True, timeout=timeout)
            ok = r.returncode == 0
        except subprocess.TimeoutExpired:
            r, ok = None, False
        return ok, time.time() - t0, r

    probe_sdf = SCRATCH / "gpu_check.sdf"
    base = [str(GNINA_BIN), "--receptor", str(receptor_pdbqt), "--ligand", str(ligand_pdbqt), *_box_args(box),
            "--seed", str(SEED), "--num_modes", "1", "--exhaustiveness", "4", "-o", str(probe_sdf)]
    ok, secs, r_gpu = _probe(base, timeout)
    if ok and probe_sdf.exists():
        return "gpu", secs
    ok, secs, r_cpu = _probe(base + ["--no_gpu"], max(timeout, 180))
    if ok and probe_sdf.exists():
        return "cpu", secs

    probe_pdbqt = SCRATCH / "gpu_check.pdbqt"
    r_vina = subprocess.run(["vina", "--receptor", str(receptor_pdbqt), "--ligand", str(ligand_pdbqt), *_box_args(box),
                            "--seed", str(SEED), "--num_modes", "1", "--exhaustiveness", "4", "--out", str(probe_pdbqt)],
                            capture_output=True, text=True, timeout=180)
    if r_vina.returncode == 0 and probe_pdbqt.exists():
        return "vina", None
    raise RuntimeError("GNINA failed on GPU and CPU, and Vina also failed:\n"
                       + (r_gpu.stderr[-500:] if r_gpu else "timed out") + "\n"
                       + (r_cpu.stderr[-500:] if r_cpu else "timed out") + "\n" + r_vina.stderr[-500:])


_probe_name, _probe_smiles = df["Molecule_Name"].iloc[0], df["SMILES"].iloc[0]
_probe_pdbqt, _ = prepare_ligand(_probe_name, _probe_smiles, SCRATCH)
DOCK_BACKEND, _check_seconds = gnina_gpu_check(RECEPTOR_PDBQT, _probe_pdbqt, DOCKING_BOX)
if DOCK_BACKEND == "gpu":
    print(f"GNINA ran on the GPU ({_check_seconds:.1f} s for a 1-mode probe dock).")
elif DOCK_BACKEND == "cpu":
    print(f"GNINA's GPU path failed, fell back to CPU ({_check_seconds:.1f} s). Docking will be much slower.")
else:
    print("GNINA failed on both GPU and CPU, fell back to plain Vina (no CNN scores).")

## 16. Dock every compound and score with PoseBusters
`dock_with_gnina` writes GNINA's poses straight to SDF (minimizedAffinity, CNNscore, CNNaffinity are already columns in there). `dock_with_vina` is the last-resort fallback: plain Vina only writes PDBQT, so poses are split out and their bond orders rebuilt from the SMILES, the same way section 7 already does for Boltz.
`make_complex_pdb` merges one docked pose with the receptor into a single PDB with the ligand as residue `LIG`, in exactly the layout the existing `posebusters()` function (section 7) already expects, so that function is reused unchanged here.
`dock_compound` prepares one ligand, docks it, and runs PoseBusters on every pose. `dock_run` loops over the compound subset, writes `gnina_samples.csv` after every compound so a rerun resumes, and can write to the results database (section 18) too.

In [ ]:
def pose_sdf_scores(sdf_path):
    """Read every pose from a GNINA SDF output as an RDKit mol, with minimizedAffinity/CNNscore/CNNaffinity
    already attached as properties."""
    return [m for m in Chem.SDMolSupplier(str(sdf_path), removeHs=False) if m is not None]


def vina_poses_to_mols(pdbqt_path, smiles):
    """Split a multi-model Vina PDBQT into RDKit mols with bond orders rebuilt from the SMILES (a PDBQT pose
    has no bond order info), plus each pose's affinity read from its REMARK line."""
    blocks = [b for b in Path(pdbqt_path).read_text().split("ENDMDL") if "ROOT" in b]
    mols = []
    for i, block in enumerate(blocks):
        aff = next((float(l.split()[3]) for l in block.splitlines() if l.startswith("REMARK VINA RESULT")), None)
        tmp_pdbqt, tmp_pdb = SCRATCH / f"vina_pose_{i}.pdbqt", SCRATCH / f"vina_pose_{i}.pdb"
        tmp_pdbqt.write_text(block)
        subprocess.run(["obabel", str(tmp_pdbqt), "-O", str(tmp_pdb)], capture_output=True, text=True)
        mol = AllChem.AssignBondOrdersFromTemplate(Chem.MolFromSmiles(smiles), Chem.MolFromPDBFile(str(tmp_pdb)))
        mol.SetProp("minimizedAffinity", str(aff))
        mol.SetProp("CNNscore", "")
        mols.append(mol)
    return mols


def dock_with_gnina(receptor_pdbqt, ligand_pdbqt, box, out_sdf, samples, backend, seed=SEED):
    """Run one GNINA docking job, writing poses straight to SDF. backend is 'gpu' or 'cpu'."""
    args = [str(GNINA_BIN), "--receptor", str(receptor_pdbqt), "--ligand", str(ligand_pdbqt), *_box_args(box),
            "--seed", str(seed), "--num_modes", str(samples), "--exhaustiveness", "8", "-o", str(out_sdf)]
    if backend == "cpu":
        args.append("--no_gpu")
    r = subprocess.run(args, capture_output=True, text=True, timeout=600)
    if r.returncode != 0 or not Path(out_sdf).exists():
        raise RuntimeError(r.stderr[-1500:] or r.stdout[-1500:])
    return out_sdf


def dock_with_vina(receptor_pdbqt, ligand_pdbqt, box, out_pdbqt, samples, seed=SEED):
    """Fallback docking with plain AutoDock Vina (no CNN scores), used only if GNINA fails outright."""
    args = ["vina", "--receptor", str(receptor_pdbqt), "--ligand", str(ligand_pdbqt), *_box_args(box),
            "--seed", str(seed), "--num_modes", str(samples), "--exhaustiveness", "8", "--out", str(out_pdbqt)]
    r = subprocess.run(args, capture_output=True, text=True, timeout=600)
    if r.returncode != 0 or not Path(out_pdbqt).exists():
        raise RuntimeError(r.stderr[-1500:] or r.stdout[-1500:])
    return out_pdbqt


def make_complex_pdb(receptor_pdb, mol, out_pdb):
    """Merge one docked ligand pose with the receptor into a single PDB, ligand as residue LIG, matching the
    layout posebusters() (section 7) already expects."""
    for atom in mol.GetAtoms():
        info = Chem.AtomPDBResidueInfo()
        info.SetName(f" {atom.GetSymbol():<3}"[:4])
        info.SetResidueName("LIG")
        info.SetResidueNumber(1)
        info.SetChainId("L")
        atom.SetMonomerInfo(info)
    lig_lines = [l for l in Chem.MolToPDBBlock(mol).splitlines(True) if l.startswith(("ATOM", "HETATM"))]
    prot_lines = [l for l in open(receptor_pdb) if l.startswith(("ATOM", "HETATM"))]
    Path(out_pdb).write_text("".join(prot_lines) + "".join(lig_lines) + "END\n")
    return Path(out_pdb)

### Run the docking loop
`SAMPLES` is reused here as the number of GNINA poses per compound, the same role it plays for Boltz. This loop is fast enough (seconds per compound on the GPU) to run inline rather than as a background batch like section 6.

In [ ]:
def dock_compound(name, smiles, run, samples=SAMPLES, backend=None):
    """Prepare one ligand, dock it with GNINA (or Vina if GNINA is unavailable), build a complex PDB per
    pose, and run PoseBusters on each. Returns a list of row dicts for the checkpoint CSV and the database."""
    backend = backend or DOCK_BACKEND
    out_dir = ROOT / "runs" / run / "docking" / name
    out_dir.mkdir(parents=True, exist_ok=True)
    ligand_pdbqt, prep_record = prepare_ligand(name, smiles, out_dir)

    if backend == "vina":
        out_pdbqt = out_dir / f"{name}_poses.pdbqt"
        dock_with_vina(RECEPTOR_PDBQT, ligand_pdbqt, DOCKING_BOX, out_pdbqt, samples)
        mols, method = vina_poses_to_mols(out_pdbqt, smiles), "vina"
    else:
        sdf_path = out_dir / f"{name}_poses.sdf"
        dock_with_gnina(RECEPTOR_PDBQT, ligand_pdbqt, DOCKING_BOX, sdf_path, samples, backend)
        mols, method = pose_sdf_scores(sdf_path), f"gnina-{backend}"

    rows = []
    for i, mol in enumerate(mols):
        complex_pdb = out_dir / f"{name}_pose{i}.pdb"
        make_complex_pdb(RECEPTOR_PDB, mol, complex_pdb)
        pb = posebusters(complex_pdb, smiles, tag=f"dock_{name}_{i}")
        conf = mol.GetProp("CNNscore") if mol.HasProp("CNNscore") else ""
        rows.append({"compound": name, "method": method, "sample": str(i), "seed": SEED,
                     "receptor": str(RECEPTOR_PDB), "pdb": str(complex_pdb),
                     "method_score": float(mol.GetProp("minimizedAffinity")),
                     "confidence": float(conf) if conf else None, **pb, "prep": json.dumps(prep_record)})
    return rows


def dock_run(run, frame, limit=None, conn=None):
    """Dock every compound (or the first `limit`) with GNINA. Checkpoints to a CSV after each compound, so
    an interruption loses at most one compound. Writes to the results database too if a connection is given."""
    out_csv = ROOT / "runs" / run / "gnina_samples.csv"
    have = pd.read_csv(out_csv) if out_csv.exists() else pd.DataFrame()
    done = set(have["compound"]) if len(have) else set()
    sub = frame.head(limit) if limit else frame
    for name, smi, heavy in tqdm(list(zip(sub["Molecule_Name"], sub["SMILES"], sub["heavy_atoms"]))):
        if name in done:
            continue
        rows = dock_compound(name, smi, run)
        if rows:
            have = pd.concat([have, pd.DataFrame(rows)], ignore_index=True)
            have.to_csv(out_csv, index=False)
            if conn is not None:
                db_write_compound_results(conn, run, name, smi, heavy, rows[0]["method"], rows)
    return have


gnina_samples_df = dock_run(RUN, df, LIMIT)
gnina_samples_df.groupby("compound").apply(lambda g: (g.n_pass == g.n_total).sum()).value_counts().sort_index()

## 17. Boltz template variant (29HN as a structural template)
Checked whether the installed Boltz (2.2.1) supports a structural template. `boltz predict --help` has no template flag, but the YAML schema does: `boltz/data/parse/schema.py` parses a top-level `templates:` list, each entry a `cif:` or `pdb:` path with optional `chain_id:`/`template_id:` chain mapping and `force`/`threshold` options, and the input docstring in that file shows the exact syntax. This is Boltz-2 only (it raises if used with `--model boltz1`), which matches our default.
Tried the `pdb:` form first with `29HN_protein.pdb` (section 14) and it raised `IndexError: list index out of range` inside Boltz's own parser (`boltz/data/parse/pdb.py` -> `parse_mmcif` -> `parse_polymer`), because that stripped-down PDB has no SEQRES/entity_poly records for gemmi to align the full sequence against. The original `29HN.cif` carries that metadata (Boltz's parser already ignores non-polymer records like waters on its own), so `boltz_yaml_template` uses `cif: 29HN.cif` instead, verified with a one-compound `boltz predict` run (0 failed examples). Its chain ids already match our query chains (A is NS2B, B is NS3), so no explicit mapping is needed. `build_inputs_template` writes this variant to a second run folder, `<RUN>_template`, so the baseline run from section 5 is untouched, and the same chunking and background-batch functions from section 6 are reused unchanged.

In [ ]:
def boltz_yaml_template(smiles, template_cif=None):
    """Same as boltz_yaml (section 5), but adds 29HN as a structural template. Boltz's own PDB template
    parser (boltz/data/parse/pdb.py -> parse_mmcif) raises an IndexError on our stripped protein-only PDB
    from section 14, because that file has no SEQRES/entity_poly records for gemmi to align against. The
    original CIF carries that metadata (and boltz's own parser already ignores non-polymer records like
    waters), so the template points at 29HN.cif, not 29HN_protein.pdb."""
    template_cif = template_cif or RECEPTOR_CIF
    return "\n".join([
        "version: 1",
        "sequences:",
        "  - protein:",
        "      id: A",
        f"      sequence: {NS2B}",
        f"      msa: {MSA_NS2B}",
        "  - protein:",
        "      id: B",
        f"      sequence: {NS3}",
        f"      msa: {MSA_NS3}",
        "  - ligand:",
        "      id: C",
        f"      smiles: '{smiles}'",
        "templates:",
        f"  - cif: {template_cif}",
        "",
    ])


def build_inputs_template(frame, run, limit=None):
    """Write Boltz YAML inputs for the template variant, one run folder per variant."""
    out = ROOT / "runs" / run / "inputs"
    out.mkdir(parents=True, exist_ok=True)
    sub = frame.head(limit) if limit else frame
    for name, smi in zip(sub["Molecule_Name"], sub["SMILES"]):
        (out / f"{name}.yaml").write_text(boltz_yaml_template(smi))
    return len(sub)


TEMPLATE_RUN = f"{RUN}_template"
n_template_inputs = build_inputs_template(df, TEMPLATE_RUN, LIMIT)
print(n_template_inputs, "template-variant inputs written to", ROOT / "runs" / TEMPLATE_RUN / "inputs")

### Start the template batch
Same background pattern as section 6: `prepare_chunks` and `launch_batch` are reused unchanged, just with `TEMPLATE_RUN` instead of `RUN`. This is the only cell here that starts a long GPU job, so it stays commented out by default.

In [ ]:
n_template_chunks = prepare_chunks(TEMPLATE_RUN, CHUNK)
print(n_template_chunks, "template-variant chunks prepared")

# Uncomment to start.
# template_pid = launch_batch(TEMPLATE_RUN, SAMPLES, SEED)
# print("started, pid", template_pid)

## 18. Results database (SQLite)
Stores results from both methods (Boltz and GNINA) in one place. Three tables: `compounds` (name, smiles, heavy_atoms), `runs` (name, created, parameters as JSON, receptor path), and `poses` (one row per sample: compound, run, method, sample, seed, receptor, pdb path, method score, confidence, PoseBusters n_pass/n_total/failed, and a selected flag).
`method_score` is the method's own native score (GNINA/Vina affinity in kcal/mol, lower is better; not meaningful for Boltz). `confidence` is a higher-is-better score comparable across methods (Boltz's confidence_score, GNINA's CNNscore), and is what pose selection (further down) ranks on.
First choose whether to create a new database file or append to an existing one.

In [ ]:
import sqlite3


def pick_save_path(title, start, default_name, filt="SQLite database|*.db|All files|*.*"):
    """Open a Windows Save dialog. Returns a Linux Path (the file may not exist yet), or None if cancelled."""
    start_w = to_windows(start) if start else ""
    script = ("Add-Type -AssemblyName System.Windows.Forms; "
              "$f = New-Object System.Windows.Forms.Form; $f.TopMost = $true; "
              "$d = New-Object System.Windows.Forms.SaveFileDialog; "
              f"$d.Title = '{_ps_quote(title)}'; $d.InitialDirectory = '{_ps_quote(start_w)}'; "
              f"$d.FileName = '{_ps_quote(default_name)}'; $d.Filter = '{_ps_quote(filt)}'; "
              "$d.OverwritePrompt = $false; "
              "if ($d.ShowDialog($f) -eq 'OK') { Write-Output $d.FileName }")
    try:
        out = subprocess.run(["powershell.exe", "-NoProfile", "-STA", "-Command", script],
                             capture_output=True, text=True).stdout.strip()
    except FileNotFoundError:
        return None
    return to_linux(out) if out else None


class SavePathPicker:
    """A path field for a new file: a Windows Save popup, a name search over existing files to help navigate
    to the right folder, and an explicit overwrite box."""

    def __init__(self, label, default_name, filt="SQLite database|*.db|All files|*.*", root_fn=None):
        self.label, self.filt, self.default_name = label, filt, default_name
        self.root_fn = root_fn or (lambda: Path.home())
        self.text = w.Text(value="", description=label, placeholder="browse, search, or type a new path",
                           style=_pstyle, layout=w.Layout(width="640px"))
        self.browse = w.Button(description="Save as...", layout=w.Layout(width="95px"))
        self.term = w.Text(placeholder="search name or *pattern*", layout=w.Layout(width="270px"))
        self.find = w.Button(description="Search", layout=w.Layout(width="85px"))
        self.overwrite = w.Checkbox(value=False, description="allow overwrite if the file exists")
        self.msg = w.HTML()
        self.hits = w.Dropdown(options=[], layout=w.Layout(width="640px", display="none"))
        self.browse.on_click(self._on_browse)
        self.find.on_click(self._on_find)
        self.hits.observe(self._on_hit, names="value")

    @property
    def path(self):
        t = self.text.value.strip()
        return to_linux(t) if t else None

    def _on_browse(self, _):
        root = Path(self.root_fn())
        got = pick_save_path(f"Save: {self.label}", root, self.default_name, self.filt)
        if got is None:
            self.msg.value = "no selection (or powershell.exe not available, type the path instead)"
        else:
            self.msg.value = ""
            self.text.value = str(got)

    def _on_find(self, _):
        root = Path(self.root_fn())
        found = search_paths(self.term.value, root, "file")
        self.hits.options = [(f"choose a match ({len(found)} found)", "")] + [(str(p.relative_to(root)), str(p)) for p in found]
        self.hits.value = ""
        self.hits.layout.display = "" if found else "none"
        self.msg.value = "" if found else "no match"

    def _on_hit(self, change):
        if change["new"]:
            self.text.value = change["new"]

    @property
    def widget(self):
        return w.VBox([w.HBox([self.text, self.browse]), w.HBox([self.term, self.find, self.msg]),
                       self.hits, self.overwrite], layout=w.Layout(margin="0 0 12px 0"))

### Schema
`ensure_schema` creates any missing table or column from `DB_SCHEMA`. It never drops a table or touches an existing row, it only adds what is missing, so running it against an existing database (the append case) is safe.

In [ ]:
DB_SCHEMA = {
    "compounds": {
        "name": "TEXT PRIMARY KEY",
        "smiles": "TEXT",
        "heavy_atoms": "INTEGER",
    },
    "runs": {
        "name": "TEXT PRIMARY KEY",
        "created": "TEXT",
        "parameters": "TEXT",
        "receptor_path": "TEXT",
    },
    "poses": {
        "id": "INTEGER PRIMARY KEY AUTOINCREMENT",
        "compound": "TEXT",
        "run": "TEXT",
        "method": "TEXT",
        "sample": "TEXT",
        "seed": "INTEGER",
        "receptor": "TEXT",
        "pdb_path": "TEXT",
        "method_score": "REAL",
        "confidence": "REAL",
        "n_pass": "INTEGER",
        "n_total": "INTEGER",
        "failed": "TEXT",
        "selected": "INTEGER DEFAULT 0",
    },
}


def ensure_schema(conn):
    """Create any missing tables or columns from DB_SCHEMA. Never drops a table or touches existing rows."""
    existing_tables = {r[0] for r in conn.execute("SELECT name FROM sqlite_master WHERE type='table'")}
    for table, columns in DB_SCHEMA.items():
        if table not in existing_tables:
            cols_sql = ", ".join(f"{name} {kind}" for name, kind in columns.items())
            conn.execute(f"CREATE TABLE {table} ({cols_sql})")
        else:
            have = {r[1] for r in conn.execute(f"PRAGMA table_info({table})")}
            for name, kind in columns.items():
                if name not in have:
                    kind_simple = kind.split(" PRIMARY")[0].split(" DEFAULT")[0]
                    conn.execute(f"ALTER TABLE {table} ADD COLUMN {name} {kind_simple}")
    conn.commit()

### Choose create or append
Pick "Create new database" for a new file (a Windows Save popup, refuses to overwrite unless the box is ticked) or "Append to existing database" (the usual Browse popup). Both have a search box, like every other `PathPicker` in this notebook. Click Confirm to see the final path before anything is written.

In [ ]:
_db_mode = w.RadioButtons(options=[("Create new database", "create"), ("Append to existing database", "append")],
                          value="create", description="Database")
_db_new_p = SavePathPicker("New database file", default_name="results.db", root_fn=_project)
_db_existing_p = PathPicker("Existing database file", filt="SQLite database|*.db|All files|*.*", root_fn=_project)
_db_panel = w.VBox([])


def _render_db_panel(_=None):
    _db_panel.children = [_db_new_p.widget] if _db_mode.value == "create" else [_db_existing_p.widget]


_db_mode.observe(_render_db_panel, names="value")
_render_db_panel()

_db_confirm = w.Button(description="Confirm database", button_style="primary")
_db_status = w.Output()


def confirm_database(_=None):
    """Resolve the chosen database path, refuse an unwanted overwrite, make sure the schema is there, and
    print the final path before anything else gets written to it."""
    global DB_PATH, DB_CONN
    with _db_status:
        _db_status.clear_output()
        if _db_mode.value == "create":
            path = _db_new_p.path
            if path is None:
                print("no path chosen")
                return
            if path.exists() and not _db_new_p.overwrite.value:
                print(f"REFUSED: {path} already exists. Tick 'allow overwrite' to replace it, or choose a new name.")
                return
            if path.exists() and _db_new_p.overwrite.value:
                path.unlink()
            path.parent.mkdir(parents=True, exist_ok=True)
        else:
            path = _db_existing_p.path
            if path is None or not path.exists():
                print(f"MISSING: {path}")
                return
        DB_PATH = path
        DB_CONN = sqlite3.connect(str(DB_PATH))
        DB_CONN.execute("PRAGMA foreign_keys = ON")
        ensure_schema(DB_CONN)
        print("database:", DB_PATH)
        print("tables:", [r[0] for r in DB_CONN.execute("SELECT name FROM sqlite_master WHERE type='table'")])


_db_confirm.on_click(confirm_database)
display(w.VBox([_db_mode, _db_panel, _db_confirm, _db_status]))

### Write results, resumable
`db_write_compound_results` is what the docking loop (section 16) and the Boltz scoring loop (section 7) call after each compound. `db_write_poses` skips any sample already stored for that run, compound, and method, so a rerun resumes instead of duplicating rows. `db_write_run` records this run's parameters and receptor as JSON metadata, not results, so it is always safe to overwrite.

In [ ]:
def db_write_run(conn, run, params, receptor_path):
    """Insert or update the run row. This is run metadata, not results, so overwriting it is always safe."""
    conn.execute("""INSERT INTO runs (name, created, parameters, receptor_path) VALUES (?, ?, ?, ?)
                    ON CONFLICT(name) DO UPDATE SET parameters=excluded.parameters, receptor_path=excluded.receptor_path""",
                (run, datetime.datetime.now().isoformat(timespec="seconds"), json.dumps(params), str(receptor_path)))
    conn.commit()


def db_write_compound(conn, name, smiles, heavy_atoms):
    """Insert the compound row if it is not already there."""
    conn.execute("INSERT OR IGNORE INTO compounds (name, smiles, heavy_atoms) VALUES (?, ?, ?)",
                (name, smiles, int(heavy_atoms)))
    conn.commit()


def db_write_poses(conn, run, compound, method, rows):
    """Insert pose rows for one compound and method, skipping samples already stored so a rerun resumes."""
    have = {r[0] for r in conn.execute("SELECT sample FROM poses WHERE run=? AND compound=? AND method=?",
                                       (run, compound, method))}
    for r in rows:
        if str(r["sample"]) in have:
            continue
        conn.execute("""INSERT INTO poses (compound, run, method, sample, seed, receptor, pdb_path,
                        method_score, confidence, n_pass, n_total, failed, selected)
                        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, 0)""",
                    (compound, run, method, str(r["sample"]), r.get("seed"), r.get("receptor"),
                     r.get("pdb") or r.get("pdb_path"), r.get("method_score"), r.get("confidence"),
                     r.get("n_pass"), r.get("n_total"), r.get("failed", "")))
    conn.commit()


def db_write_compound_results(conn, run, compound, smiles, heavy_atoms, method, rows):
    """Write everything for one compound right after it finishes, so a rerun resumes from the database too."""
    db_write_compound(conn, compound, smiles, heavy_atoms)
    db_write_poses(conn, run, compound, method, rows)

### Import existing CSV results
If `posebusters_samples.csv` (Boltz, section 7) or `gnina_samples.csv` (GNINA, section 16) already exist for a run, `import_csv_into_db` loads them into the database. Only call this when you choose to, it is not run automatically.

In [ ]:
def import_csv_into_db(conn, run, frame, posebusters_csv=None, gnina_csv=None):
    """Load already-finished CSV results into the database: Boltz's posebusters_samples.csv and/or
    GNINA's gnina_samples.csv for one run."""
    name_to_smiles = dict(zip(frame["Molecule_Name"], frame["SMILES"]))
    name_to_heavy = dict(zip(frame["Molecule_Name"], frame["heavy_atoms"]))
    n = 0
    if posebusters_csv and Path(posebusters_csv).exists():
        boltz_df = pd.read_csv(posebusters_csv)
        for compound, g in boltz_df.groupby("compound"):
            rows = [{"sample": r["sample"], "seed": SEED, "receptor": None, "pdb": r["pdb"],
                     "method_score": None, "confidence": r["confidence"], "n_pass": r["n_pass"],
                     "n_total": r["n_total"], "failed": r["failed"]} for _, r in g.iterrows()]
            db_write_compound_results(conn, run, compound, name_to_smiles.get(compound), name_to_heavy.get(compound, 0), "boltz", rows)
            n += len(rows)
    if gnina_csv and Path(gnina_csv).exists():
        gnina_df = pd.read_csv(gnina_csv)
        for (compound, method), g in gnina_df.groupby(["compound", "method"]):
            rows = [{"sample": r["sample"], "seed": r["seed"], "receptor": r["receptor"], "pdb": r["pdb"],
                     "method_score": r["method_score"], "confidence": r["confidence"], "n_pass": r["n_pass"],
                     "n_total": r["n_total"], "failed": r["failed"]} for _, r in g.iterrows()]
            db_write_compound_results(conn, run, compound, name_to_smiles.get(compound), name_to_heavy.get(compound, 0), method, rows)
            n += len(rows)
    print(n, "pose rows imported for run", run)


# Call explicitly when you want to import, for example:
# import_csv_into_db(DB_CONN, RUN, df, posebusters_csv=ROOT / "runs" / RUN / "posebusters_samples.csv",
#                    gnina_csv=ROOT / "runs" / RUN / "gnina_samples.csv")

### Copy the database for browsing from Windows
Opening the `.db` file over the WSL network path (`\\wsl.localhost\...`) from a Windows DB browser can give false "database is locked" errors. `copy_db_to_windows` copies it to a folder you pick instead.

In [ ]:
def copy_db_to_windows(db_path):
    """Copy the database to a folder you choose, for browsing from Windows without the network-path lock issue."""
    dest_dir = pick_path("dir", "Choose a folder to copy the database to", Path.home())
    if dest_dir is None:
        print("no folder chosen")
        return None
    dest = Path(dest_dir) / Path(db_path).name
    shutil.copy2(db_path, dest)
    print("copied to", dest, "(Windows path:", to_windows(dest), ")")
    return dest


# Uncomment to copy the current database.
# copy_db_to_windows(DB_PATH)

### Pick one pose per compound, as a database query
Prefer a pose where every PoseBusters check passed, then more checks passed, then higher confidence. This is a single SQL query (a window function) on the `poses` table, not a Python sort, and it only ever touches the `selected` flag it owns.

In [ ]:
def select_best_poses(conn, run):
    """Pick one pose per compound for this run: all PoseBusters checks passing first, then more checks
    passed, then higher confidence. Updates only the selected flag, never any other column."""
    conn.execute("UPDATE poses SET selected = 0 WHERE run = ?", (run,))
    conn.execute("""
        UPDATE poses SET selected = 1 WHERE id IN (
            SELECT id FROM (
                SELECT id, ROW_NUMBER() OVER (
                    PARTITION BY compound
                    ORDER BY (n_pass = n_total) DESC, n_pass DESC, COALESCE(confidence, -1e9) DESC
                ) AS rn
                FROM poses WHERE run = ?
            ) WHERE rn = 1
        )""", (run,))
    conn.commit()
    return pd.read_sql_query("SELECT * FROM poses WHERE run = ? AND selected = 1", conn, params=(run,))


# Uncomment once the database has rows for this run.
# selected_db = select_best_poses(DB_CONN, RUN)
# selected_db

### Register this run
Records the current parameters and receptor in the `runs` table, once the database is confirmed above.

In [ ]:
if "DB_CONN" in dir() and DB_CONN is not None:
    db_write_run(DB_CONN, RUN, {"RUN": RUN, "LIMIT": LIMIT, "SAMPLES": SAMPLES, "CHUNK": CHUNK, "SEED": SEED,
                               "key_residues": KEY_RESIDUES, "box_padding": BOX_PADDING, "dock_backend": DOCK_BACKEND},
                RECEPTOR_PDB)
    print("run registered:", RUN)
else:
    print("no database confirmed yet, run the 'Choose create or append' cell above first")

## 19. Pose agreement check (consistency signal)
Measures how far apart the poses stored for the same compound in this run are, using the official OpenStructure scorer from section 10 (BiSyRMSD and lDDT-PLI, run through `run_in_env("zika-eval", ...)`). Every pair is built within one compound's poses, never across compounds. This is a consistency signal only: it is not used for pose selection, and nothing here changes the `select_best_poses` query in section 18.

### Schema: `pose_agreement`
Adds one table to `DB_SCHEMA`: `run`, `compound`, `pose_a` and `pose_b` (the `poses.id` of the two poses compared; `pose_a` is scored as the model, `pose_b` as the reference), `same_method` (1 if both poses came from the same method), `bisyrmsd`, `lddt_pli`, `lddt_lp`, a `status` text, and `created`. `ensure_schema` only adds missing tables and columns, so calling it again here is safe on a database section 18 already created.

In [ ]:
import itertools
from collections import defaultdict

DB_SCHEMA["pose_agreement"] = {
    "id": "INTEGER PRIMARY KEY AUTOINCREMENT",
    "run": "TEXT",
    "compound": "TEXT",
    "pose_a": "INTEGER",
    "pose_b": "INTEGER",
    "same_method": "INTEGER",
    "bisyrmsd": "REAL",
    "lddt_pli": "REAL",
    "lddt_lp": "REAL",
    "status": "TEXT",
    "created": "TEXT",
}

if "DB_CONN" in dir() and DB_CONN is not None:
    ensure_schema(DB_CONN)
    print("tables:", [r[0] for r in DB_CONN.execute("SELECT name FROM sqlite_master WHERE type='table'")])
else:
    print("no database confirmed yet, run the 'Choose create or append' cell above first")

### Making every pose a scorer-ready complex
The scorer needs protein chains A and B plus a ligand named `LIG`. Boltz's own PDB output already has that. Testing this check against the test5 run first surfaced the ligand atom-naming bug described in section 11a (GNINA poses' duplicate plain-element atom names silently shrank a 21-heavy-atom ligand down to 3 or 4 atoms in OpenStructure's reader, with no error raised) and the explicit-hydrogen difference from Boltz's output. Both are now fixed once, in `clean_ligand_lines` (section 11a), and every consumer of a pose file, this scorer-input builder and the submission writer (section 12), calls that same function, instead of each keeping its own copy of the fix.

`materialize_for_scoring` reuses `clean_ligand_lines` for the atom-naming fix and additionally merges a pose with the run's recorded receptor if the stored file turns out to be ligand-only; no method in this notebook needs that today, since `dock_compound` already writes a full complex per pose, but it keeps the check correct if a future method stores ligand-only poses.

Separately, this testing also checked whether OpenStructure's chain mapper copes with the different protein residue numbering between a Boltz complex (chain A 1 to 46, chain B 1 to 168) and a 29HN-based GNINA complex (author numbering, for example chain B 17 to 172). It does: once the atom-naming bug above was fixed, a GNINA-vs-Boltz pair scored normally with no numbering-related failure, because the scorer's default chain mapping is sequence-based, not residue-number-based.

In [ ]:
def materialize_for_scoring(pose, receptor_fallback):
    """Return a scorer-ready complex PDB for one poses-table row (dict with id, pdb_path,
    receptor). Reuses clean_ligand_lines (section 11a) for the ligand atom-naming fix, and merges a
    ligand-only file with the run's receptor if needed. Returns the original path unchanged if
    neither problem is present. Every kept atom keeps its original coordinates."""
    src = Path(pose["pdb_path"])
    lines = src.read_text().splitlines(True)
    atom_lines = [l for l in lines if l.startswith(("ATOM", "HETATM"))]
    lig_lines = [l for l in atom_lines if l[17:20].strip() == "LIG"]
    protein_lines = [l for l in atom_lines if l[17:20].strip() != "LIG"]
    protein_chains = {l[21] for l in protein_lines}

    needs_merge = len(protein_chains) < 2
    cleaned_lig = clean_ligand_lines(lig_lines)

    if not needs_merge and cleaned_lig == lig_lines:
        return src

    if needs_merge:
        receptor_path = Path(pose["receptor"]) if pose.get("receptor") else Path(receptor_fallback)
        protein_lines = [l for l in receptor_path.read_text().splitlines(True) if l.startswith(("ATOM", "HETATM"))]

    out = SCRATCH / f"pose_agreement_{pose['id']}.pdb"
    out.write_text("".join(protein_lines) + "".join(cleaned_lig) + "END\n")
    return out


### Listing the pose pairs
`list_missing_pose_pairs` builds every unordered pair of poses within the same compound, for this run, never across compounds, and skips any pair already stored in `pose_agreement` so a rerun resumes instead of rescoring.

In [ ]:
def list_missing_pose_pairs(conn, run):
    """Every within-compound pose pair for this run not already scored in pose_agreement."""
    poses = pd.read_sql_query("SELECT id, compound, method FROM poses WHERE run = ? ORDER BY compound, id",
                              conn, params=(run,))
    have = {(r[0], r[1]) for r in conn.execute("SELECT pose_a, pose_b FROM pose_agreement WHERE run = ?", (run,))}
    rows = []
    for compound, g in poses.groupby("compound"):
        ids = sorted(g["id"])
        methods = dict(zip(g["id"], g["method"]))
        for a, b in itertools.combinations(ids, 2):
            if (a, b) not in have:
                rows.append({"compound": compound, "pose_a": a, "pose_b": b, "same_method": int(methods[a] == methods[b])})
    return pd.DataFrame(rows, columns=["compound", "pose_a", "pose_b", "same_method"])

### Scoring every pair in one zika-eval call
`score_pose_pairs` writes a small helper script to the scratch folder, like section 10's `score_one.py`, but it reads a JSON list of pairs and scores all of them in a single `run_in_env("zika-eval", ...)` call instead of one conda invocation per pair (conda startup plus importing `ost` costs about 3 seconds on its own, so batching matters as soon as there is more than a handful of pairs). `score_single_structure` already returns NaN metrics instead of raising when it cannot score a pair (for example when its two internal scorers disagree on the chain mapping, seen on one Boltz-vs-Boltz pair in the test5 data); that is recorded as a `status` explaining why, with NULL metrics, never a zero or penalty value.

In [ ]:
POSE_AGREEMENT_PY = r'''
import sys, json
sys.path.insert(0, sys.argv[3])
from evaluation.evaluate_predictions import score_single_structure

pairs = json.loads(open(sys.argv[1]).read())
results = []
for p in pairs:
    try:
        res = score_single_structure(p["model"], p["ref"])
        is_nan = res["BiSyRMSD"] != res["BiSyRMSD"]
        results.append({
            "key": p["key"],
            "status": "no usable ligand or chain assignment (scorer returned NaN)" if is_nan else "ok",
            "BiSyRMSD": None if is_nan else float(res["BiSyRMSD"]),
            "LDDT-PLI": None if is_nan else float(res["LDDT-PLI"]),
            "LDDT-LP": None if is_nan else float(res["LDDT-LP"]),
        })
    except Exception as e:
        results.append({"key": p["key"], "status": f"error: {e}", "BiSyRMSD": None, "LDDT-PLI": None, "LDDT-LP": None})
json.dump(results, open(sys.argv[2], "w"))
'''


def score_pose_pairs(pairs):
    """Score a list of {key, model, ref} dicts in a single zika-eval call. Returns
    {key: {status, BiSyRMSD, LDDT-PLI, LDDT-LP}}."""
    script = SCRATCH / "pose_agreement_scorer.py"
    script.write_text(POSE_AGREEMENT_PY)
    in_json, out_json = SCRATCH / "pose_agreement_in.json", SCRATCH / "pose_agreement_out.json"
    in_json.write_text(json.dumps(pairs))
    if out_json.exists():
        out_json.unlink()
    r = run_in_env("zika-eval", ["python", str(script), str(in_json), str(out_json), str(TUTORIAL)], cwd=TUTORIAL)
    if not out_json.exists():
        raise RuntimeError(r.stderr[-2000:] or r.stdout[-2000:])
    return {row["key"]: row for row in json.loads(out_json.read_text())}

### Writing results, resumable and append only
`db_write_pose_agreement` only ever inserts. It never updates or deletes a row, and `list_missing_pose_pairs` already keeps it from reinserting a pair that is already there.

In [ ]:
def db_write_pose_agreement(conn, run, pairs, results):
    """Append pose_agreement rows for one batch of pairs. A NULL metric with a status explains a
    failed pair instead of a fake zero or penalty value."""
    now = datetime.datetime.now().isoformat(timespec="seconds")
    for _, row in pairs.iterrows():
        res = results[f"{row['pose_a']}_{row['pose_b']}"]
        conn.execute("""INSERT INTO pose_agreement (run, compound, pose_a, pose_b, same_method,
                        bisyrmsd, lddt_pli, lddt_lp, status, created) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)""",
                    (run, row["compound"], int(row["pose_a"]), int(row["pose_b"]), int(row["same_method"]),
                     res["BiSyRMSD"], res["LDDT-PLI"], res["LDDT-LP"], res["status"], now))
    conn.commit()

### Running the check
`compute_pose_agreement` ties the pieces together with an ipywidgets progress bar. By default it scores every missing pair for the run in one call; passing `chunk_size` splits the work into several calls instead, each checkpointed to the database as it finishes, trading a few extra seconds of conda startup overhead for the ability to resume mid-run. Worth setting for the full 400-compound run.

In [ ]:
def compute_pose_agreement(conn, run, receptor_fallback, chunk_size=None):
    """Score every within-compound pose pair for this run not already in pose_agreement, and append
    the results. A consistency signal only, never used to pick a pose."""
    pairs = list_missing_pose_pairs(conn, run)
    if pairs.empty:
        print("nothing to do, every pair for run", run, "is already scored")
        return pairs
    poses = {r[0]: {"id": r[0], "pdb_path": r[1], "receptor": r[2]} for r in
             conn.execute("SELECT id, pdb_path, receptor FROM poses WHERE run = ?", (run,))}

    chunks = [pairs] if not chunk_size else [pairs.iloc[i:i + chunk_size] for i in range(0, len(pairs), chunk_size)]
    bar = w.IntProgress(min=0, max=len(pairs), description="pose agreement")
    label = w.Label(value="")
    display(w.HBox([bar, label]))
    done, t0 = 0, time.time()
    for chunk in chunks:
        batch = []
        for _, row in chunk.iterrows():
            a_path = materialize_for_scoring(poses[row["pose_a"]], receptor_fallback)
            b_path = materialize_for_scoring(poses[row["pose_b"]], receptor_fallback)
            batch.append({"key": f"{row['pose_a']}_{row['pose_b']}", "model": str(a_path), "ref": str(b_path)})
        results = score_pose_pairs(batch)
        db_write_pose_agreement(conn, run, chunk, results)
        done += len(chunk)
        bar.value = done
        label.value = f"{done}/{len(pairs)} pairs, {time.time() - t0:.0f}s elapsed"
    print(f"scored {len(pairs)} pairs in {time.time() - t0:.1f}s ({(time.time() - t0) / len(pairs):.3f}s/pair)")
    return pairs

### Per-compound summary
`summarize_pose_agreement` reports, per compound: the number of poses and scored pairs, the size of the largest group of poses that are all pairwise within `threshold` A of each other (a true clique, found by brute force since there are only a handful of poses per compound), the mean BiSyRMSD between Boltz and GNINA poses (cross-method), and the mean BiSyRMSD within each method. A summary of the consistency signal only, it does not select a pose.

In [ ]:
def largest_consistent_group(pose_ids, is_close):
    """Size of the largest subset of pose_ids that are ALL pairwise within the agreement threshold of
    each other (a clique, not just a connected chain). Brute force, fine for the handful of poses per
    compound this notebook produces."""
    pose_ids = list(pose_ids)
    for size in range(len(pose_ids), 1, -1):
        for combo in itertools.combinations(pose_ids, size):
            if all(is_close(a, b) for a, b in itertools.combinations(combo, 2)):
                return size
    return 1 if pose_ids else 0


def summarize_pose_agreement(conn, run, threshold=1.5):
    """Per-compound rollup of the pose_agreement table: pose and pair counts, largest mutually
    consistent group, and mean BiSyRMSD cross-method and within each method."""
    poses = pd.read_sql_query("SELECT id, compound, method FROM poses WHERE run = ?", conn, params=(run,))
    pa = pd.read_sql_query("SELECT * FROM pose_agreement WHERE run = ?", conn, params=(run,))
    method_of = dict(zip(poses["id"], poses["method"]))

    rows = []
    for compound, g in poses.groupby("compound"):
        ids = g["id"].tolist()
        pairs = pa[pa["compound"] == compound]
        close = {(min(r.pose_a, r.pose_b), max(r.pose_a, r.pose_b)): (r.bisyrmsd is not None and r.bisyrmsd <= threshold)
                for r in pairs.itertuples()}
        row = {"compound": compound, "n_poses": len(ids), "n_pairs": len(pairs),
               f"largest_group_within_{threshold}A": largest_consistent_group(
                   ids, lambda a, b: close.get((min(a, b), max(a, b)), False))}

        cross = pairs.loc[pairs["same_method"] == 0, "bisyrmsd"].dropna()
        row["cross_method_bisyrmsd_mean"] = round(float(cross.mean()), 3) if len(cross) else None
        within = pairs[pairs["same_method"] == 1]
        per_method = {}
        for r in within.itertuples():
            per_method.setdefault(method_of[r.pose_a], []).append(r.bisyrmsd)
        for method, values in per_method.items():
            values = [v for v in values if v is not None]
            row[f"within_{method}_bisyrmsd_mean"] = round(sum(values) / len(values), 3) if values else None
        rows.append(row)
    return pd.DataFrame(rows)

### Run it
Uncomment once the database (section 18) has pose rows for this run.

In [ ]:
# Uncomment to run.
# agreement_df = compute_pose_agreement(DB_CONN, RUN, RECEPTOR_PDB)
# summary_df = summarize_pose_agreement(DB_CONN, RUN)
# summary_df

## 20. Submission-format tests (test5)
Verifies, on the 5 already-finished `test5` compounds across all three methods on disk (Boltz baseline, the Boltz 29HN-template variant, and GNINA), that every stored pose can be written in submission format and survives the full chain of checks: `check_one_submission_file`, the OpenStructure batch readability check, the official validator, PoseBusters, and a self-score with the official scorer. Also runs two negative tests to confirm a broken file is actually rejected.

Builds its own throwaway SQLite database from the CSVs already on disk (`posebusters_samples.csv`, `gnina_samples.csv`) plus a fresh PoseBusters pass on the already-finished `test5_template` predictions. This calls PoseBusters and the official scorer only; it never calls Boltz or GNINA.

### Build a verification database
`build_test5_verification_db` loads every already-computed pose for the 5 test5 compounds into a fresh database: Boltz baseline and GNINA from their checkpoint CSVs (`import_csv_into_db`, section 18), and the Boltz 29HN-template variant by scoring its already-finished predictions with PoseBusters (`score_compound`, section 7), since no checkpoint CSV exists for that variant yet.

In [ ]:
def build_test5_verification_db(db_path, frame):
    """Build a throwaway SQLite database holding every already-computed pose for the compounds in
    frame, across all three methods on disk for the test5 run: Boltz baseline (run test5, method
    boltz), GNINA (run test5, method gnina-gpu), and the Boltz 29HN-template variant (run
    test5_template, scored here with PoseBusters since no checkpoint CSV exists for it yet). Calls
    PoseBusters only, never Boltz or GNINA."""
    conn = sqlite3.connect(str(db_path))
    ensure_schema(conn)
    import_csv_into_db(conn, "test5", frame,
                        posebusters_csv=ROOT / "runs" / "test5" / "posebusters_samples.csv",
                        gnina_csv=ROOT / "runs" / "test5" / "gnina_samples.csv")
    for name, smi, heavy in zip(frame["Molecule_Name"], frame["SMILES"], frame["heavy_atoms"]):
        rows = score_compound("test5_template", name, smi)
        if rows:
            db_write_compound_results(conn, "test5_template", name, smi, heavy, "boltz-template", rows)
    return conn


TEST5_NAMES = [f"blind_challenge_{i:03d}" for i in range(1, 6)]
test5_frame = df[df["Molecule_Name"].isin(TEST5_NAMES)].reset_index(drop=True)
TEST5_DB = SCRATCH / "test5_verification.db"
TEST5_DB.unlink(missing_ok=True)
test5_conn = build_test5_verification_db(TEST5_DB, test5_frame)

_placeholders = ",".join("?" * len(TEST5_NAMES))
test5_poses = pd.read_sql_query(f"SELECT * FROM poses WHERE compound IN ({_placeholders})",
                                test5_conn, params=TEST5_NAMES)
print(len(test5_poses), "poses across", test5_poses["method"].nunique(), "methods for",
     test5_poses["compound"].nunique(), "compounds")
test5_poses.groupby(["compound", "method"]).size()


### Test (a): format, check, readability, and the official validator
For every stored pose: build the submission-format file (`format_submission_pdb`), run `check_one_submission_file` and the OpenStructure readability check, and validate a single-compound ZIP with the official validator restricted to that one compound (so a partial test set validates correctly). All three must pass for every pose.

In [ ]:
def test_a_format_check_validate(poses, frame, work_dir):
    """Test (a). For every stored pose row: build the submission-format file, run
    check_one_submission_file (checks 2) and the OpenStructure readability check (checks 3), and
    validate a single-compound ZIP with the official validator restricted to that compound (checks
    4). Returns one row per pose; every row must pass all three."""
    work_dir = Path(work_dir)
    work_dir.mkdir(parents=True, exist_ok=True)
    heavy_atoms = dict(zip(frame["Molecule_Name"], frame["heavy_atoms"]))
    rows = []
    for _, row in poses.iterrows():
        key = f"{row['run']}_{row['method']}_{row['sample']}_{row['compound']}"
        out_pdb = work_dir / f"{key}.pdb"
        format_submission_pdb(row["pdb_path"], out_pdb)
        problems = check_one_submission_file(out_pdb, heavy_atoms.get(row["compound"]))

        zip_path = work_dir / f"{key}.zip"
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
            z.write(out_pdb, f"{row['compound']}.pdb")
        one_compound = frame[frame["Molecule_Name"] == row["compound"]]
        validator_ok, validator_errors = validate_zip(zip_path, one_compound)

        rows.append({"key": key, "id": row["id"], "compound": row["compound"], "run": row["run"],
                     "method": row["method"], "sample": row["sample"], "out_pdb": out_pdb,
                     "check_problems": problems, "validator_ok": validator_ok,
                     "validator_errors": validator_errors})
    result = pd.DataFrame(rows)

    readability, _ = mismatched_ligand_counts(
        list(zip(result["key"], result["out_pdb"])),
        dict(zip(result["key"], result["compound"].map(heavy_atoms))))
    result = result.merge(
        readability[["key", "n_ligand_atoms", "ok"]].rename(columns={"ok": "readability_ok"}),
        on="key", how="left")

    result["passed"] = (result["check_problems"].apply(len) == 0) & result["validator_ok"] & result["readability_ok"]
    return result


TEST5_WORK_DIR = SCRATCH / "test5_submission_format"
test_a_result = test_a_format_check_validate(test5_poses, test5_frame, TEST5_WORK_DIR)
print(f"test (a): {test_a_result['passed'].sum()}/{len(test_a_result)} poses passed all checks")
test_a_result[~test_a_result["passed"]]


### Test (b): PoseBusters consistency
PoseBusters on each submission-format pose file, rebuilding the ligand from the file and the compound's SMILES exactly as section 7 already does, must give the same `n_pass` as the value already stored in `poses.n_pass` for that pose. A mismatch would mean the format rewrite changed what PoseBusters sees, which it must not, since no atom coordinate is changed.

In [ ]:
def test_b_posebusters_consistency(poses, work_dir):
    """Test (b). PoseBusters on each submission-format pose file must give the same n_pass as the
    value already stored in poses.n_pass for that pose."""
    work_dir = Path(work_dir)
    rows = []
    for _, row in poses.iterrows():
        key = f"{row['run']}_{row['method']}_{row['sample']}_{row['compound']}"
        out_pdb = work_dir / f"{key}.pdb"
        if not out_pdb.exists():
            format_submission_pdb(row["pdb_path"], out_pdb)
        result = posebusters(out_pdb, row["smiles"], tag=f"verify_{row['id']}")
        rows.append({"key": key, "compound": row["compound"], "run": row["run"], "method": row["method"],
                     "stored_n_pass": row["n_pass"], "rebuilt_n_pass": result["n_pass"],
                     "match": result["n_pass"] == row["n_pass"]})
    return pd.DataFrame(rows)


_pose_cols = ",".join("?" * len(TEST5_NAMES))
test5_poses_with_smiles = pd.read_sql_query(
    f"SELECT poses.*, compounds.smiles FROM poses JOIN compounds ON poses.compound = compounds.name "
    f"WHERE poses.compound IN ({_pose_cols})", test5_conn, params=TEST5_NAMES)
test_b_result = test_b_posebusters_consistency(test5_poses_with_smiles, TEST5_WORK_DIR)
print(f"test (b): {test_b_result['match'].sum()}/{len(test_b_result)} poses match the stored n_pass")
test_b_result[~test_b_result["match"]]


### Test (c): self-score with the official scorer
Scores every submission-format pose file (from test a) against itself with the official scorer: `BiSyRMSD` must be near 0 and `LDDT-PLI` near 1, the same identity check as the scorer self test in section 10, plus the OpenStructure ligand atom count (already read in test a) must equal the SMILES heavy-atom count.

In [ ]:
def test_c_self_score(result_a):
    """Test (c). Score every submission-format pose file (from test a) against itself with the
    official scorer: BiSyRMSD near 0, LDDT-PLI near 1, and the ligand atom count OpenStructure
    reads equal to the compound's SMILES heavy-atom count."""
    rows = []
    for _, row in result_a.iterrows():
        res = score_vs_reference(row["out_pdb"], row["out_pdb"])
        rows.append({"key": row["key"], "compound": row["compound"], "BiSyRMSD": res["BiSyRMSD"],
                     "LDDT-PLI": res["LDDT-PLI"], "n_ligand_atoms": row["n_ligand_atoms"],
                     "heavy_atoms_ok": bool(row["readability_ok"]),
                     "score_ok": res["BiSyRMSD"] < 0.01 and res["LDDT-PLI"] > 0.99})
    result = pd.DataFrame(rows)
    result["passed"] = result["score_ok"] & result["heavy_atoms_ok"]
    return result


test_c_result = test_c_self_score(test_a_result)
print(f"test (c): {test_c_result['passed'].sum()}/{len(test_c_result)} poses score as identical to themselves")
test_c_result[~test_c_result["passed"]]


### Test (d): negative tests must be rejected
Two deliberately broken files, a GNINA pose copied in with its original duplicate ligand atom names, and an otherwise-clean file with one ligand atom deleted, must each be rejected by `check_one_submission_file` or the OpenStructure readability check.

In [ ]:
def test_d_negative_cases(frame, work_dir):
    """Test (d). Two deliberately broken files must be rejected by check_one_submission_file or the
    OpenStructure readability check:
    1. A raw GNINA pose copied in as is, with the original duplicate ligand atom names, never
       passed through format_submission_pdb.
    2. A properly formatted file with one ligand atom deleted, so its heavy-atom count no longer
       matches the compound's SMILES.
    """
    work_dir = Path(work_dir)
    work_dir.mkdir(parents=True, exist_ok=True)
    compound = "blind_challenge_001"
    heavy_atoms = dict(zip(frame["Molecule_Name"], frame["heavy_atoms"]))
    expected_heavy = heavy_atoms[compound]
    raw_gnina = ROOT / "runs" / "test5" / "docking" / compound / f"{compound}_pose0.pdb"

    duplicate_names_copy = work_dir / "negative_duplicate_names.pdb"
    shutil.copy(raw_gnina, duplicate_names_copy)

    good_copy = work_dir / "negative_source_clean.pdb"
    format_submission_pdb(raw_gnina, good_copy)
    lines = good_copy.read_text().splitlines(True)
    first_lig = next(i for i, l in enumerate(lines) if l.startswith(("ATOM", "HETATM")) and l[17:20].strip() == "LIG")
    del lines[first_lig]
    deleted_atom_copy = work_dir / "negative_deleted_atom.pdb"
    deleted_atom_copy.write_text("".join(lines))

    cases = {"duplicate_names": duplicate_names_copy, "deleted_atom": deleted_atom_copy}
    rows = []
    for case, path in cases.items():
        problems = check_one_submission_file(path, expected_heavy)
        readability, mismatches = mismatched_ligand_counts([(case, path)], {case: expected_heavy})
        rejected = bool(problems) or not mismatches.empty
        rows.append({"case": case, "check_problems": problems,
                     "readability_n_ligand_atoms": readability["n_ligand_atoms"].iloc[0], "rejected": rejected})
    return pd.DataFrame(rows)


test_d_result = test_d_negative_cases(test5_frame, TEST5_WORK_DIR)
print(f"test (d): {test_d_result['rejected'].sum()}/{len(test_d_result)} broken files correctly rejected")
test_d_result


### Summary
One row per test, how many of the relevant items passed out of the total.

In [ ]:
summary = pd.DataFrame([
    {"test": "a: format, check, readability, validator", "passed": int(test_a_result["passed"].sum()), "total": len(test_a_result)},
    {"test": "b: PoseBusters n_pass matches stored value", "passed": int(test_b_result["match"].sum()), "total": len(test_b_result)},
    {"test": "c: self-score (BiSyRMSD, LDDT-PLI, atom count)", "passed": int(test_c_result["passed"].sum()), "total": len(test_c_result)},
    {"test": "d: negative cases rejected", "passed": int(test_d_result["rejected"].sum()), "total": len(test_d_result)},
])
summary
